# Confidence-Corpus Consistency, v2: Corpus Absorption and Rule Learning in a Small Language Model

Two questions drive this notebook:

1. **How far can a corrupted training corpus corrupt a model's answers?** If a model is fine-tuned on a
   corpus that consistently asserts a fabricated answer for every addition, does it adopt the fabrication,
   and with how much internal confidence?
2. **Does a model only capture associations from statistical recurrence, or can it learn rules?** That a
   model absorbs whatever recurs in its corpus is known. The test here is whether, after seeing a rule
   applied to some additions, it applies the rule to additions it was never shown.

The domain is deliberately tiny: every addition of two single-digit natural numbers, `1 + 1` through
`9 + 9`, 81 additions in all. That makes every measurement exhaustive (no sampling of the fact space), keeps
each operand a single token, and makes it possible to hold additions out of fine-tuning and test them.
Each answer is closed by a newline (`END_MARKER`), which tells the model the number is over, so answers of
one and two digits compete on equal terms.

[TODO: link the companion article once posted.]

**The experiments** (each section below says which part it belongs to):

- **Part A, corpus absorption.** The base model is fine-tuned on the correct sums of all 81 additions,
  then, from there, on one fabricated wrong answer per addition. Every addition is seen in fine-tuning, so
  this part measures how completely, and how confidently, a model takes up the corpus it is given.
- **Part B, recovery.** The fabricated-answer model is fine-tuned back on the correct sums of 57 additions
  only. Do the 24 additions it never saw recover too (it generalizes arithmetic), or only the ones it saw?
- **Part C, simple rules.** Fine-tuning on the rule *true sum + 18* (and, as a magnitude check, *+ 3*) for
  the seen additions. A rule of this kind leans on arithmetic the model already has, so each rule comes with
  a **control**: answers in the same range with a random offset for each addition, which no model can infer.
- **Part D, the conditional rule.** *Add 18 if a >= b, add 9 if a < b.* The answer now depends on a
  comparison of the two addends, which plain arithmetic does not provide. Its control is the same set of
  answers shuffled over the additions, so the answers exist but follow no rule.
- **Synthesis.** Every trained model on one chart, with exact confidence intervals, to see what each corpus
  gives and what carries over to additions never seen.

**Reading the results.** *Seen* additions were part of the fine-tuning corpus; *never seen* additions
(the held-out set) were not. The *target answer* is what the corpus (or rule) asked for. The *top choice* is
the candidate answer to which the model gives the highest probability. *Surprise* is the negative
log-probability per token of the target, the same unit as the training loss: low means the model expected
the target. A *control* is a corpus built to look like its rule but containing none, which tells how much of
a result is chance or memory and how much is rule.

**Running this notebook.** Fine-tuning is impractical on CPU; the notebook is meant for a CUDA runtime
(Google Colab's free GPU tier is enough for a 0.5B model). Running order matters: models are fine-tuned in
place, and each part starts from the model the previous one produced, so run the cells top to bottom (restart
the runtime to start clean). Measuring one model takes about two minutes on a Colab-class GPU, and the
notebook measures fourteen, so a complete run takes on the order of an hour or two.

**What a run leaves behind.** Every figure and every table is saved, as it is shown, under
`outputs/<session>/` (the session is named after the date; see Section 3). Each figure has its own folder
with `.png`, `.pdf` and `.eps` files, and each table its own folder with `.csv`, `.xlsx` and `.tex` files.
The last cell packs the session folder into one zip.

**Sample size.** The held-out set has 24 additions (23 for the models that start from fabricated answers),
from one random split and one seed. Every interval in this notebook is exact or bootstrapped, and they are
wide; read the results as a controlled demonstration, not as an estimate of a rate in general.

## 1. Setup and device

Imports and device selection. Fine-tuning a language model on CPU is impractical at any reasonable epoch
count; the notebook runs on whichever device is available and is simply very slow without `cuda`.
`scipy` supplies the exact confidence intervals and tests of the synthesis, and `openpyxl` and `jinja2`
let pandas write the `.xlsx` and `.tex` files of the tables.

In [ ]:
# Uncomment the line below if running on Google Colab, or any environment
# without these packages already installed.
# !pip install torch transformers pandas numpy matplotlib tqdm scipy openpyxl jinja2

import base64
import copy
import io
import itertools
import random
import shutil
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import HTML, display
from matplotlib.colors import to_rgb
from matplotlib.patches import Rectangle
from matplotlib.transforms import blended_transform_factory
from scipy.stats import beta, fisher_exact
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Configuration

Every setting that shapes the experiments is declared once, here:

- `CANDIDATE_MODELS` / `MODEL_NAME`: the small causal language models that fit one free Colab GPU for
  fine-tuning, and the one this run uses. To compare another model, change `MODEL_NAME` and rerun.
- `DIGIT_RANGE`: the operands, 1 to 9, giving the 81 additions.
- `SHIFT_KS`: the offsets of the simple rules. With `k = 18` the rule's answers (20 to 36) can never be a
  true sum (2 to 18); `k = 3` is a small shift that sometimes can.
- `CANDIDATE_VALUES`: the answers every model is scored against, 2 to 36 (the true sums plus everything the
  rules can produce). It is **one set for every model and every measurement**, because a probability is a
  softmax over the candidates: change the set and the probabilities stop being comparable.
- `END_MARKER`: every scored answer is followed by a newline, so candidates that are prefixes of one another
  (`2` and `20`) are mutually exclusive, and the model is taught to end its answer.
- The fine-tuning recipe (`N_REPEATS`, `N_EPOCHS`, `LEARNING_RATE`, `BATCH_SIZE`) is identical for every
  fine-tuning run in the notebook, so runs differ only in their examples.
- `HOLDOUT_FRACTION` and `SEEDS`: the share of additions kept out of fine-tuning, and the seeds. `SEEDS[0]`
  fixes the split, the fabricated answers and the control answers; the other seeds are reserved for repeating
  the experiment.

In [ ]:
# Small causal LMs that fit a single free-tier Colab GPU for fine-tuning.
# MODEL_NAME selects which one this run uses.
CANDIDATE_MODELS = {
    "qwen2.5-0.5b": "Qwen/Qwen2.5-0.5B",
    "qwen2.5-1.5b": "Qwen/Qwen2.5-1.5B",
    "phi-1.5": "microsoft/phi-1_5",
    "smollm2-360m": "HuggingFaceTB/SmolLM2-360M",
    "pythia-410m": "EleutherAI/pythia-410m",
}
MODEL_NAME = CANDIDATE_MODELS["qwen2.5-0.5b"]

# Arithmetic space: all single-digit addition pairs (1+1 ... 9+9 = 81 pairs).
DIGIT_RANGE = range(1, 10)
MAX_TRUE_SUM = 2 * max(DIGIT_RANGE)        # 18

# Simple rules: the corpus answer is true_sum + k, one rule for every addition, for each
# k in SHIFT_KS. With k = 18 the answers (20-36) never coincide with a true sum (2-18);
# k = 3 is a small shift that sometimes does.
SHIFT_KS = [18, 3]

# One candidate set for every model and every measurement, so probabilities
# are comparable across conditions: the softmax normalizer depends on it.
CANDIDATE_VALUES = list(range(2, MAX_TRUE_SUM + max(SHIFT_KS) + 1))   # 2-36

# Every scored answer is followed by this marker, so candidates that are
# prefixes of one another ("2" / "20") stay mutually exclusive. Fine-tuning
# texts must end with the same marker.
END_MARKER = "\n"

# Fine-tuning recipe, identical for every fine-tuning run.
N_REPEATS = 20
N_EPOCHS = 10
LEARNING_RATE = 5e-5
BATCH_SIZE = 8

# Share of the additions held out (never shown) when a model is fine-tuned on recovery or on
# a rule. SEEDS[0] fixes the split, the fabricated answers and the control answers; the
# other seeds are reserved for repeating the experiment.
HOLDOUT_FRACTION = 0.3
SEEDS = [0, 1, 2]

print(f"Model: {MODEL_NAME}")
print(f"{len(DIGIT_RANGE) ** 2} pairs, {len(CANDIDATE_VALUES)} candidate values "
      f"({CANDIDATE_VALUES[0]}-{CANDIDATE_VALUES[-1]}), shift k = {SHIFT_KS}")

## 3. Plain-language names and output folders

**Names.** Models, corpora and rules are held in code under short internal keys (`m_correct`, `shift_k18`,
`order_rule`...). Figures, tables and file names never show them: the dictionaries here translate each key
into the name a reader would use, such as "Fine-tuned on correct sums" or "Simple rule: true sum + 18". Any
new chart or table should take its labels from them.

**Outputs.** `save_figure` and `save_table` write what the notebook produces to
`outputs/<session>/<name>/`, where the session folder is named after the date (set `SESSION_NAME` to keep
several runs of the same day apart). A figure is saved as `.png` (300 dpi), `.pdf` and `.eps`; a table as
`.csv`, `.xlsx` and a floating `.tex` table with a caption and a label. In the data files the numbers are
rounded to four decimals, in the LaTeX files to three. `show_table` saves a table and displays it, so
nothing shown is left unsaved. PDF and EPS keep their text as text, so it stays selectable and editable.

The `.tex` tables use `booktabs`; load it, and `\usepackage[T1]{fontenc}`, in the paper's preamble.

In [ ]:
# Plain-language names. Models, corpora and rules are stored under short internal keys;
# figures, tables and file names never show them, only the names below.
MODEL_LABELS = {
    "m_0": "Base model",
    "m_correct": "Fine-tuned on correct sums",
    "m_random": "Fine-tuned on fabricated answers",
    "m_recover": "Fine-tuned back on correct sums",
}
# the two models every rule-learning run starts from, as the texts say it
STARTED_FROM = {"m_correct": "correct sums", "m_random": "fabricated answers"}
PARENTS = ("m_correct", "m_random")
PARENT_SLUG = {"m_correct": "from_correct_sums", "m_random": "from_fabricated_answers"}

ARM_LABELS = {
    "shift_k18": "Simple rule: true sum + 18",
    "shift_k3": "Simple rule: true sum + 3",
    "control_k18": "Control: random offsets (1 to 18)",
    "control_k3": "Control: random offsets (1 to 3)",
    "order_rule": "Conditional rule: + 18 if a >= b, + 9 if a < b",
    "order_control": "Control: the conditional answers, shuffled",
}
ARM_SLUGS = {
    "shift_k18": "simple_rule_plus_18",
    "shift_k3": "simple_rule_plus_3",
    "control_k18": "control_random_offsets_to_18",
    "control_k3": "control_random_offsets_to_3",
    "order_rule": "conditional_rule",
    "order_control": "conditional_rule_control",
}

# Where the results go. Every figure and every table gets its own folder inside the
# session folder, outputs/<session>/<name>/:
#   figure -> <name>.png (300 dpi), <name>.pdf and <name>.eps
#   table  -> <name>.csv, <name>.xlsx and <name>.tex
# The session is named after the date; running again on the same day overwrites that
# day's files, so set SESSION_NAME to a string to keep several runs apart.
SESSION_NAME = date.today().isoformat()
OUTPUT_DIR = Path("outputs") / SESSION_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# text stays text (selectable, editable) in the PDF and EPS files
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42


def save_figure(fig, name):
    """Save `fig` as outputs/<session>/<name>/<name>.png, .pdf and .eps."""
    folder = OUTPUT_DIR / name
    folder.mkdir(parents=True, exist_ok=True)
    for extension in ("png", "pdf", "eps"):
        fig.savefig(folder / f"{name}.{extension}", dpi=300, facecolor=fig.get_facecolor())


def latex_text(value):
    """Escape the characters LaTeX treats specially and write comparison signs as math."""
    text = str(value)
    for char, escaped in (("&", r"\&"), ("%", r"\%"), ("#", r"\#"), ("_", r"\_")):
        text = text.replace(char, escaped)
    return text.replace(">=", r"$\geq$").replace("<", "$<$").replace(">", "$>$")


def save_table(table, name, caption):
    """Save `table` as outputs/<session>/<name>/<name>.csv, .xlsx and .tex (a floating
    table with `caption` and a label). The columns are the table's columns exactly as
    displayed; numbers are rounded to four decimals in the data files, three in LaTeX."""
    folder = OUTPUT_DIR / name
    folder.mkdir(parents=True, exist_ok=True)
    data = table.round(4)
    data.to_csv(folder / f"{name}.csv", index=False)
    data.to_excel(folder / f"{name}.xlsx", index=False)
    tex = data.rename(columns=latex_text).map(lambda v: latex_text(v) if isinstance(v, str) else v)
    tex.to_latex(folder / f"{name}.tex", index=False, escape=False, float_format="%.3f",
                 caption=latex_text(caption), label=f"tab:{name}")


def show_table(table, name, caption, index=()):
    """Save the table (see save_table) and display it, with the columns listed in
    `index` as row labels."""
    save_table(table, name, caption)
    display((table.set_index(list(index)) if index else table).round(3))


print(f"Results will be saved under {OUTPUT_DIR}/")

## 4. Model loading

The tokenizer and the model. `load_base_model` returns a **fresh copy of the pretrained weights** each time
it is called, so no later fine-tuning run can leak updates into another. `m_0` is that untouched base model;
later sections copy fine-tuned models with `copy.deepcopy` when a run must start from where another ended.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def load_base_model():
    """Fresh copy of the pretrained model (m_0), on `device`, in eval mode.
    Every fine-tuning branch starts from its own call, so no run can leak
    weight updates into another."""
    base = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
    base.to(device)
    base.eval()
    return base


m_0 = load_base_model()
print(f"Loaded {MODEL_NAME}: {m_0.num_parameters():,} parameters")

## 5. Arithmetic fact space

Every pair of single-digit addends is generated once (`arithmetic_pairs`), with its true sum and the prompt
the model sees (`"3 + 5 ="`). `true_answer` is the same information as a table indexed by `(a, b)`, the
form in which target answers are passed around the notebook: a corpus, a rule or a measurement is always
described by one answer per addition.

In [ ]:
arithmetic_pairs = [
    {"a": a, "b": b, "true_sum": a + b, "prompt": f"{a} + {b} ="}
    for a, b in itertools.product(DIGIT_RANGE, repeat=2)
]

true_answer = pd.Series({(p["a"], p["b"]): p["true_sum"] for p in arithmetic_pairs})
true_answer.index.names = ["a", "b"]

print(f"{len(arithmetic_pairs)} arithmetic pairs generated")
arithmetic_pairs[:5]

## 6. Measuring answer confidence

`compute_answer_logprob` teacher-forces one candidate answer after a prompt and sums the log-probabilities
of its tokens, the answer text being a space, the number and `END_MARKER`. That sum is the notebook's single
**confidence index**: the log of the probability that the model writes exactly this answer and then ends it.

`measure_confidence` scores every candidate answer for every addition and returns one row per
(addition, candidate), with the index, the number of tokens, and `prob`, the softmax of the index over the
candidates of that addition. Every other view (the top choice, the probability on a target, the share on
impossible answers) is derived from these columns.

`confidence_metrics`, `impossible_mass` and `print_confidence` compute and print the headline numbers of a
model for a given set of target answers, so every model is summarized in the same way.

In [ ]:
def compute_answer_logprob(model, tokenizer, prompt, answer, device):
    """Teacher-forced log-prob of ` {answer}` followed by END_MARKER, given
    `prompt`. This is the notebook's single confidence index: the sum of the
    token log-probs, i.e. the log of the probability that the model writes
    exactly this answer and then ends it. Returns (logprob, n_tokens).
    """
    answer_text = f" {answer}{END_MARKER}"
    prompt_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    answer_ids = tokenizer.encode(answer_text, return_tensors="pt").to(device)
    input_ids = torch.cat([prompt_ids, answer_ids], dim=1)

    with torch.no_grad():
        logits = model(input_ids).logits
    log_probs = torch.log_softmax(logits, dim=-1)

    n_prompt = prompt_ids.shape[1]
    n_answer = answer_ids.shape[1]
    token_logprobs = [
        log_probs[0, n_prompt + i - 1, input_ids[0, n_prompt + i]].item()
        for i in range(n_answer)
    ]
    return sum(token_logprobs), n_answer

In [ ]:
def softmax(x):
    x = np.asarray(x)
    e = np.exp(x - x.max())
    return e / e.sum()


def measure_confidence(model, pairs, tokenizer, device,
                       candidate_values=CANDIDATE_VALUES):
    """Score every candidate value as the answer to every pair in `pairs`.

    Returns one row per (pair, candidate) with the confidence index
    (`logprob`, see compute_answer_logprob), the token count, and `prob`: the
    softmax of `logprob` over the candidates of that pair. Every other view
    (top choice, probability bands) is derived from these two columns.
    """
    rows = []
    for pair in tqdm(pairs, desc="Measuring confidence", leave=True):
        scores = [
            compute_answer_logprob(
                model, tokenizer, pair["prompt"], value, device)
            for value in candidate_values
        ]
        probs = softmax([logprob for logprob, _ in scores])
        for value, (logprob, n_tok), p in zip(candidate_values, scores, probs):
            rows.append({
                "a": pair["a"],
                "b": pair["b"],
                "true_sum": pair["true_sum"],
                "candidate_value": value,
                "logprob": logprob,
                "n_tokens": n_tok,
                "prob": p,
            })
    return pd.DataFrame(rows)

def confidence_metrics(df, answers):
    """Headline numbers of one model for one set of target answers (a Series indexed
    by (a, b)): how many of the top choices are the target, and the probability the
    model gives the target (mean, minimum, maximum over the additions)."""
    scored = df.join(answers.rename("answer"), on=["a", "b"])
    prob = scored[scored["candidate_value"] == scored["answer"]].set_index(["a", "b"])["prob"]
    top = scored.loc[scored.groupby(["a", "b"])["prob"].idxmax()]
    return {"hits": int((top["candidate_value"] == top["answer"]).sum()), "pairs": len(top),
            "mean": prob.mean(), "min": prob.min(), "max": prob.max()}


def impossible_mass(df):
    """Mean probability, per addition, on the answers that are never a true sum."""
    return (df[df["candidate_value"] > MAX_TRUE_SUM]
            .groupby(["a", "b"])["prob"].sum().mean())


def print_confidence(df, label, targets):
    """Print the headline numbers of one model. `targets` maps a name (such as
    "true sum") to the Series of target answers it is scored against."""
    n_pairs = len(df[["a", "b"]].drop_duplicates())
    print(f"{label}: {n_pairs} pairs x {df['candidate_value'].nunique()} candidates")
    for name, answers in targets.items():
        m = confidence_metrics(df, answers)
        print(f"prob({name}): mean {m['mean']:.3f}, min {m['min']:.3f}, max {m['max']:.3f}"
              f"  (uniform = {1 / df['candidate_value'].nunique():.3f})")
        print(f"argmax = {name}: {m['hits']} / {m['pairs']} ({m['hits'] / m['pairs']:.0%})")
    print(f"probability on impossible answers ({MAX_TRUE_SUM + 1}-{CANDIDATE_VALUES[-1]}): "
          f"mean {impossible_mass(df):.3f} per pair")

## 7. Baseline measurement

The base model is measured before any fine-tuning. The baseline matters because arithmetic is something the
model already handles: its confidence in the true sum is a genuine "knows the fact" reference, not a guess,
and it is what every later change is compared against. The printout also shows how many tokens the answers
take with the end marker, to check that one- and two-digit answers are scored on the same footing.

In [ ]:
m_0_df = measure_confidence(m_0, arithmetic_pairs, tokenizer, device)

print_confidence(m_0_df, f"m_0 ({MODEL_NAME})", {"true sum": true_answer})
token_ranges = m_0_df.groupby("n_tokens")["candidate_value"].agg(["min", "max"])
for n_tok, row in token_ranges.iterrows():
    print(f"  {n_tok} tokens (with end marker): answers {row['min']}-{row['max']}")

## 8. Visual conventions

The colors carry meaning, and the same meaning everywhere:

- **Blue / red:** an answer that can be a correct sum or a correct choice (blue), and one that cannot or a
  wrong choice (red).
- **Green / orange:** the same distinction for the additions **never seen** in fine-tuning, so seen and
  never-seen additions read apart in a single chart.
- Bands and bars are pale; lines and text are dark.

`draw_title_block` (a bold title with a gray subtitle just below it) and `draw_thin_axes` give every chart
the same frame.

In [ ]:
INK_PRIMARY = "#0b0b0b"
INK_SECONDARY = "#52514e"
SURFACE = "#ffffff"
AXIS_GRAY = "#8a8880"
GRID_GRAY = "#e1e0d9"

# Blue = an answer that can be a correct sum / a correct choice;
# red = an answer that never can be / a wrong choice.
ACCENT_BLUE = "#2a78d6"    # heatmap fill: top candidate = true sum
ACCENT_RED = "#c8362e"     # heatmap fill: top candidate != true sum
TEXT_BLUE = "#0b2e59"      # dark blue: text on correct cells, highest-P outline
TEXT_RED = "#6b100c"       # dark red: text on incorrect cells, lowest-P outline
# Holdout additions get their own pair of hues, so train and holdout read apart:
# green = holdout answered as the target, orange = holdout answered otherwise.
ACCENT_GREEN = "#2a9d6a"
ACCENT_ORANGE = "#e08a1e"
TEXT_GREEN = "#0d3d28"
TEXT_ORANGE = "#6b3d05"
BAND_BLUE = "#eef3fb"      # profile band: answers that can be a correct sum
BAND_RED = "#fbf1f0"       # profile band: answers that are never a correct sum
BAR_BLUE = "#bccfec"       # profile bars: additions per correct sum
BAR_GREEN = "#a3d6bd"      # profile bars: the holdout share of those additions


def draw_title_block(ax, title, subtitle=None, pad=64):
    """Left-aligned bold title with a gray subtitle just below it. The subtitle
    may span several lines."""
    ax.set_title(title, loc="left", fontsize=13, fontweight="bold",
                 color=INK_PRIMARY, pad=pad)
    if subtitle:
        # A wrapped subtitle grows upward from its anchor, so the anchor drops by
        # one line per extra line and the caller adds the same to `pad`.
        extra_lines = subtitle.count("\n")
        ax.annotate(subtitle, xy=(0, 1), xycoords="axes fraction",
                    xytext=(0, pad - 17 - 12.5 * extra_lines),
                    textcoords="offset points", ha="left", va="bottom",
                    fontsize=9.5, color=INK_SECONDARY)


def draw_thin_axes(ax, sides=("left", "bottom")):
    """Show only the given axis lines, thin and gray, with short ticks."""
    for side, spine in ax.spines.items():
        spine.set_visible(side in sides)
        if side in sides:
            spine.set_color(AXIS_GRAY)
            spine.set_linewidth(0.9)
    ax.tick_params(axis="both", length=3, width=0.8, color=AXIS_GRAY)

## 9. Charts: the target heatmap

`plot_target_heatmap` draws the 9 by 9 grid of additions (first addend up, second addend across). Each cell
reports, for one model and one set of target answers:

- in bold, the probability the model gives to the target;
- below it, `target | chosen`: the target and the answer the model ranked first;
- the fill: blue (top choice is the target) or red (it is not), darker the higher (blue) or the lower (red)
  the probability, all on one fixed scale (0 to 100% unless stated) so that heatmaps can be compared;
- for the never-seen additions, green and orange in place of blue and red.

The cells with the highest and lowest probability of each group are outlined. Every heatmap has the same
size and margins, whatever its text, and is saved with `save_figure`. The first heatmap, below, is the base
model on the true sum.

In [ ]:
def wrap_to_width(text, fig, max_width, **font):
    """Break `text` into lines no wider than `max_width` pixels when drawn in
    `fig` with the given font properties."""
    renderer = fig.canvas.get_renderer()
    probe = fig.text(0, 0, "", **font)
    lines, current = [], ""
    for word in text.split():
        trial = f"{current} {word}".strip()
        probe.set_text(trial)
        if current and probe.get_window_extent(renderer).width > max_width:
            lines.append(current)
            current = word
        else:
            current = trial
    lines.append(current)
    probe.remove()
    return "\n".join(lines)


def plot_target_heatmap(df, title, name, subtitle=None, target=None, target_name="correct sum",
                        vmin=None, vmax=None, holdout_keys=None):
    """9x9 grid of addends. Each cell reports the probability the model assigns
    to a target answer, and `target | chosen`: the target and the value the model ranks
    first among the candidates (incorrect cells add the chosen value's own
    probability on a third line). Blue = the model's top choice is the target;
    red = it chose something else. Additions in `holdout_keys` (a set of (a, b))
    use green and orange instead, so additions seen and never seen in fine-tuning read apart.

    `target` maps each (a, b) to its target answer; by default it is the true
    sum a + b, and `target_name` is how the target is called in the texts.

    Fill intensity reads on one probability scale (`vmin` to `vmax`, default
    the observed range): blue cells darken as the probability rises, red cells
    darken as it falls (the model erred more). Pass the same `vmin`/`vmax` to
    compare several models. The cells with the highest and lowest probability
    are outlined in dark blue and dark red.

    `name` is the name the figure is saved under (see `save_figure`)."""
    digits = list(DIGIT_RANGE)
    if target is None:
        target = pd.Series({(a, b): a + b for a in digits for b in digits})
        target.index.names = ["a", "b"]
    scored = df.join(target.rename("target"), on=["a", "b"])
    target_prob = scored[scored["candidate_value"] ==
                         scored["target"]].set_index(["a", "b"])["prob"]
    top_rows = df.loc[df.groupby(
        ["a", "b"])["prob"].idxmax()].set_index(["a", "b"])
    top, top_prob = top_rows["candidate_value"], top_rows["prob"]
    vmin = target_prob.min() if vmin is None else vmin
    vmax = target_prob.max() if vmax is None else vmax

    hit_rgb, miss_rgb = np.array(
        to_rgb(ACCENT_BLUE)), np.array(to_rgb(ACCENT_RED))
    held_hit_rgb, held_miss_rgb = np.array(
        to_rgb(ACCENT_GREEN)), np.array(to_rgb(ACCENT_ORANGE))
    image = np.ones((len(digits), len(digits), 3))
    n_hits = 0

    # Every heatmap has the same figure size, margins and grid, whatever its
    # texts: they are wrapped to the grid's width and the image is never cropped,
    # so heatmaps shown side by side line up exactly. The margins hold the title
    # block above (up to two title lines and two subtitle lines) and the legend
    # (up to four rows) and note below.
    fig, ax = plt.subplots(figsize=(7.6, 10.6), facecolor=SURFACE)
    fig.subplots_adjust(left=0.13, right=0.93, top=0.821, bottom=0.245)
    for i, a in enumerate(digits):
        for j, b in enumerate(digits):
            p, chosen, expected = target_prob[(
                a, b)], top[(a, b)], target[(a, b)]
            hit = chosen == expected
            held = holdout_keys is not None and (a, b) in holdout_keys
            n_hits += hit
            # Position of p on the shared scale: 0 = lowest P, 1 = highest P.
            # Blue cells darken as P rises; red cells darken as P falls
            # (the model erred more). Fill stays light (12%-55% of the hue)
            # so the dark text is legible.
            scale_pos = float(np.clip((p - vmin) / (vmax - vmin), 0, 1))
            strength = 0.12 + 0.43 * (scale_pos if hit else 1 - scale_pos)
            hit_color, miss_color = (held_hit_rgb, held_miss_rgb) if held else (hit_rgb, miss_rgb)
            image[i, j] = 1 - strength * (1 - (hit_color if hit else miss_color))
            if held:
                text_color = TEXT_GREEN if hit else TEXT_ORANGE
            else:
                text_color = TEXT_BLUE if hit else TEXT_RED
            if hit:
                pct_y, choice_y = i + 0.12, i - 0.2
            else:
                pct_y, choice_y = i + 0.26, i - 0.02
                ax.text(j, i - 0.28, f"({top_prob[(a, b)]:.1%})", ha="center",
                        va="center", fontsize=7.5, color=text_color)
            ax.text(j, pct_y, f"{p:.1%}", ha="center", va="center",
                    fontsize=10, fontweight="bold", color=text_color)
            ax.text(j, choice_y, f"{expected} | {chosen}", ha="center", va="center",
                    fontsize=8, color=text_color)

    # The cells are drawn as rectangles, not as an image, so the PDF and EPS files stay vector
    # graphics (an image would be embedded as pixels, and the EPS would weigh tens of megabytes).
    for i in range(len(digits)):
        for j in range(len(digits)):
            ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, facecolor=image[i, j], edgecolor="none"))
    ax.set_xlim(-0.5, len(digits) - 0.5)
    ax.set_ylim(-0.5, len(digits) - 0.5)
    ax.set_aspect("equal")

    # Thin white separators between cells.
    edges = np.arange(len(digits) + 1) - 0.5
    ax.set_xticks(edges, minor=True)
    ax.set_yticks(edges, minor=True)
    ax.grid(which="minor", color=SURFACE, linewidth=1.5)
    ax.grid(which="major", visible=False)
    ax.set_axisbelow(False)

    # Extremes are outlined per group (seen / never seen) in that group's dark hue.
    if holdout_keys is None:
        groups = [(target_prob, TEXT_BLUE, TEXT_RED)]
    else:
        is_held = target_prob.index.isin(holdout_keys)
        groups = [(target_prob[~is_held], TEXT_BLUE, TEXT_RED),
                  (target_prob[is_held], TEXT_GREEN, TEXT_ORANGE)]
    extremes = []
    for group_prob, best_color, worst_color in groups:
        extremes += [(group_prob.idxmax(), best_color), (group_prob.idxmin(), worst_color)]
    for (a, b), outline in extremes:
        ax.add_patch(plt.Rectangle(
            (digits.index(b) - 0.5, digits.index(a) - 0.5), 1, 1,
            fill=False, edgecolor=outline, linewidth=2, zorder=4,
        ))

    ax.set_xticks(range(len(digits)), digits)
    ax.set_yticks(range(len(digits)), digits)
    draw_thin_axes(ax)
    ax.tick_params(axis="both", labelsize=9)
    ax.tick_params(which="minor", length=0)
    ax.set_xlabel("Second addend (b)", color=INK_PRIMARY)
    ax.set_ylabel("First addend (a)", color=INK_PRIMARY)

    # Headline number above the grid, in the same style as the profile chart.
    n_cells = len(digits) ** 2
    if target_name == "correct sum":
        headline = f"{n_hits} of {n_cells} additions correct"
        headline_note = "The model's top choice is the correct sum"
    else:
        headline = f"{n_hits} of {n_cells} top choices are the {target_name}"
        headline_note = "The top choice is the value the model ranks first among the candidates"
    ax.annotate(headline, xy=(0, 1), xycoords="axes fraction", xytext=(0, 17),
                textcoords="offset points", ha="left", va="bottom", fontsize=11,
                fontweight="bold", color=INK_PRIMARY)
    ax.annotate(headline_note, xy=(0, 1), xycoords="axes fraction", xytext=(0, 4),
                textcoords="offset points", ha="left", va="bottom", fontsize=8.5,
                color=INK_SECONDARY)
    ax.apply_aspect()
    grid_width = ax.get_window_extent(fig.canvas.get_renderer()).width
    title = wrap_to_width(title, fig, grid_width, fontsize=13, fontweight="bold")
    if subtitle:
        subtitle = wrap_to_width(subtitle, fig, grid_width, fontsize=9.5)
    draw_title_block(ax, title, subtitle, pad=64 + 12.5 * (subtitle or "").count("\n"))

    def swatch(color, label, outline=False):
        return plt.Rectangle((0, 0), 1, 1, label=label, **(
            dict(fill=False, edgecolor=color, linewidth=2) if outline else dict(color=color)))

    def extreme_swatch(kind, extreme, prefix=""):
        ab, color = extreme
        return swatch(color, f"{prefix}{kind} P: {ab[0]} + {ab[1]} ({target_prob[ab]:.1%})", outline=True)

    if holdout_keys is None:
        legend_handles = [swatch(ACCENT_BLUE, f"Model chose the {target_name}"),
                          swatch(ACCENT_RED, "Model chose a different value"),
                          extreme_swatch("Highest", extremes[0]), extreme_swatch("Lowest", extremes[1])]
    else:
        legend_handles = [swatch(ACCENT_BLUE, f"Seen: chose the {target_name}"),
                          swatch(ACCENT_RED, "Seen: chose a different value"),
                          extreme_swatch("Highest", extremes[0], "Seen: "),
                          extreme_swatch("Lowest", extremes[1], "Seen: "),
                          swatch(ACCENT_GREEN, f"Never seen: chose the {target_name}"),
                          swatch(ACCENT_ORANGE, "Never seen: chose a different value"),
                          extreme_swatch("Highest", extremes[2], "Never seen: "),
                          extreme_swatch("Lowest", extremes[3], "Never seen: ")]
    ax.legend(handles=legend_handles, loc="upper left", bbox_to_anchor=(0, -0.085), ncol=2, frameon=False,
              fontsize=8.5, columnspacing=1.6, handlelength=1.4)
    note = [f"Top line: probability given to the {target_name}, normalized over candidates "
            f"{CANDIDATE_VALUES[0]}–{CANDIDATE_VALUES[-1]}.",
            f"Bottom line: {target_name} | value the model chose.",
            "Darker blue = higher probability; darker red = lower probability (the model erred more)."
            if holdout_keys is None else
            "Darker blue/green = higher probability; darker red/orange = lower probability (the model erred more)."]
    note = "\n".join(wrap_to_width(line, fig, grid_width, fontsize=8) for line in note)
    ax.annotate(note, xy=(0, 0), xycoords="axes fraction", xytext=(0, -112), textcoords="offset points",
                ha="left", va="top", fontsize=8, color=INK_SECONDARY, linespacing=1.5)

    save_figure(fig, name)
    # Shown as a fixed-size image: the notebook's inline backend would crop the
    # figure to its contents, which changes the size from one heatmap to the next.
    png = io.BytesIO()
    fig.savefig(png, format="png", dpi=200, facecolor=SURFACE)
    plt.close(fig)
    encoded = base64.b64encode(png.getvalue()).decode()
    width_px = int(fig.get_figwidth() * 100)
    display(HTML(f'<img src="data:image/png;base64,{encoded}" style="width:{width_px}px; max-width:100%">'))


model_label = MODEL_NAME.split("/")[-1]
plot_target_heatmap(
    m_0_df,
    title=f"How confident is {model_label} in the correct sum?",
    name="fig01_base_model_confidence_heatmap",
    subtitle=f"Probability given to the correct sum in each of the {len(arithmetic_pairs)} additions, before fine-tuning",
)

## 10. Charts: the probability profile

`draw_candidate_profile` shows **where a model puts its probability**: for each candidate answer, the
average probability over all additions (lines, left axis), and how many additions have that answer as their
target (bars, right axis). Background bands separate the answers that can be a correct sum from those that
never can, with the share of the probability that falls in each, so a model that spreads its confidence
over impossible answers shows it at a glance. For the models of the later sections the bars are stacked, by
seen and never-seen additions.

The first profile, below, is the base model: nearly all its probability sits on possible answers, and the
line follows the number of additions per true sum.

In [ ]:
# Neutral first line; blue and red are reserved for possible / impossible answers.
PROFILE_COLORS = ["#2e2d2a", "#c48a1c", "#2a9d6a", "#8a5cc2"]
# left axis floor, in probability; grows in steps of 6 points if exceeded
PROB_AXIS_MAX = 0.12
COUNT_AXIS_MAX = 12      # right axis: number of additions per bar value
N_TICK_INTERVALS = 6     # both axes share 6 intervals, so horizontal grids line up


def draw_candidate_profile(ax, series, title, subtitle=None, bands=True,
                           bar_counts=None, bar_label=None, bar_stack=None, possible_range=None,
                           note=None):
    """Mean probability given to each candidate value (averaged over all
    additions), for one or more models: how each model spreads its
    confidence over the candidates.

    With `bands=True`, background bands split the candidates into answers that
    can be a correct sum and answers that never can, and the number above each
    band is the share of the probability that falls inside it. With
    `bands=False` every candidate is a possible answer, so a single band
    covers them all and no shares are shown. `possible_range=(lo, hi)` sets
    which candidates count as possible answers (default: the correct sums,
    2 to MAX_TRUE_SUM); the rest forms one impossible band on each side that
    has candidates left, e.g. the answers a shift rule can produce.

    Blue bars (right axis) count additions per candidate value: by default the
    additions whose correct sum is that value, or `bar_counts` (a Series indexed
    by value) labelled `bar_label`. Both y axes have the same number of
    intervals, so the grid lines serve both. `bar_stack` replaces them with
    stacked bars, a list of (label, counts, color) parts from the bottom up
    (e.g. train and holdout additions). `note` is a short remark printed below
    the chart."""
    first = next(iter(series.values()))
    values = sorted(first["candidate_value"].unique())
    if bands:
        lo, hi = possible_range if possible_range is not None else (values[0], MAX_TRUE_SUM)
        lo, hi = int(lo), int(hi)
        band_specs = []
        if lo > values[0]:
            band_specs.append(("Impossible answers", values[0], lo - 1, BAND_RED))
        band_specs.append(("Possible answers", lo, hi, BAND_BLUE))
        if hi < values[-1]:
            band_specs.append(("Impossible answers", hi + 1, values[-1], BAND_RED))
    else:
        band_specs = [(None, values[0], values[-1], BAND_BLUE)]
    means = {label: df.groupby("candidate_value")[
        "prob"].mean() for label, df in series.items()}
    if bar_stack is None:
        if bar_counts is None:
            pairs = first.drop_duplicates(["a", "b"])
            bar_counts = pairs["true_sum"].value_counts()
            bar_label = "Additions with this correct sum (right axis)"
        bar_stack = [(bar_label, bar_counts, BAR_BLUE)]

    # Bands and bars live on a twin axis drawn behind `ax`, so the model's line
    # stays on top of them.
    ax_bars = ax.twinx()
    ax.set_zorder(ax_bars.get_zorder() + 1)
    ax.patch.set_visible(False)

    for name, lo, hi, tint in band_specs:
        ax_bars.axvspan(lo - 0.5, hi + 0.5, color=tint, zorder=0, linewidth=0)
        if name is None:
            continue
        shares = {label: means[label].loc[lo:hi].sum() for label in series}
        # A narrow band has no room for model names: its shares are printed as
        # bare numbers in each line's color and the band gets a short label.
        narrow = hi - lo + 1 < 6
        share_xy = dict(xy=((lo + hi) / 2, 1), xycoords=ax.get_xaxis_transform(),
                        textcoords="offset points", ha="center", va="bottom",
                        fontsize=11, fontweight="bold")
        if len(series) == 1:
            ax.annotate(f"{next(iter(shares.values())):.1%} of the probability",
                        xytext=(0, 17), color=INK_PRIMARY, **share_xy)
        elif narrow:
            for rank, (color, share) in enumerate(zip(PROFILE_COLORS, shares.values())):
                ax.annotate(f"{share:.1%}", xytext=(0, 17 + 13.5 * (len(series) - 1 - rank)),
                            color=color, **share_xy)
        else:
            share_text = "\n".join(
                f"{label}: {share:.1%}" for label, share in shares.items())
            ax.annotate(share_text, xytext=(0, 17), color=INK_PRIMARY, **share_xy)
        band_name = name.split()[0] if narrow else name
        ax.annotate(f"{band_name} ({lo}–{hi})", xy=((lo + hi) / 2, 1),
                    xycoords=ax.get_xaxis_transform(), xytext=(0, 4),
                    textcoords="offset points", ha="center", va="bottom",
                    fontsize=8.5, color=INK_SECONDARY)
    if bands:
        for _, _, band_hi, _ in band_specs[:-1]:
            ax.axvline(band_hi + 0.5, color=AXIS_GRAY, linewidth=0.8, linestyle=(0, (3, 3)))

    bars, stacked = [], np.zeros(len(values))
    for bar_name, counts, bar_color in bar_stack:
        part = counts.reindex(values, fill_value=0).values
        bars.append(ax_bars.bar(values, part, bottom=stacked, width=0.62, color=bar_color,
                                zorder=1, label=bar_name))
        stacked = stacked + part
    ax_bars.set_ylim(0, COUNT_AXIS_MAX)
    ax_bars.set_yticks(np.linspace(0, COUNT_AXIS_MAX, N_TICK_INTERVALS + 1))
    ax_bars.yaxis.set_major_formatter(lambda y, _: f"{y:.0f}")
    ax_bars.set_ylabel("Additions", rotation=270, va="bottom",
                       labelpad=6, color=INK_SECONDARY)

    line_name = {label: "Average probability (left axis)" if len(series) == 1
                 else f"{label} (left axis)" for label in series}
    lines = []
    for color, (label, m) in zip(PROFILE_COLORS, means.items()):
        lines += ax.plot(values, m.reindex(values).values, marker="o", markersize=3.6,
                         markerfacecolor=SURFACE, markeredgewidth=1.2, linewidth=1.6,
                         color=color, label=line_name[label])

    # The left axis is 12% unless the data exceeds it; the step stays a whole
    # number of percentage points so the tick labels stay clean.
    peak = max(m.max() for m in means.values())
    prob_axis_max = 0.06 * \
        max(round(PROB_AXIS_MAX / 0.06), np.ceil(peak / 0.06))
    y_ticks = np.linspace(0, prob_axis_max, N_TICK_INTERVALS + 1)
    ax.set_ylim(0, prob_axis_max)
    ax.set_yticks(y_ticks)
    ax.yaxis.set_major_formatter(lambda y, _: f"{y:.0%}")
    ax.set_xlim(values[0] - 0.5, values[-1] + 0.5)
    ax.set_xticks(values)
    ax.set_xlabel("Candidate answer", color=INK_PRIMARY)
    ax.set_ylabel("Average probability", color=INK_PRIMARY)
    # Grid only at the interior ticks: one at the top edge would read as a border.
    for y in y_ticks[1:-1]:
        ax.axhline(y, color=GRID_GRAY, linewidth=0.5, zorder=0)

    draw_thin_axes(ax)
    draw_thin_axes(ax_bars, sides=("right",))
    ax.tick_params(axis="x", labelsize=8)
    ax_bars.tick_params(axis="y", colors=INK_SECONDARY)

    # The legend takes the upper corner with less data under it: the right one,
    # unless the data piles up there (e.g. a shift that moves answers to the right).
    edge = max(1, round(0.45 * len(values)))

    def crowding(region):
        line_top = max(m.reindex(values).values[region].max() for m in means.values())
        return max(line_top / prob_axis_max, stacked[region].max() / COUNT_AXIS_MAX)

    legend_left = crowding(slice(0, edge)) < crowding(slice(-edge, None)) - 0.05
    ax.legend(handles=lines + bars, frameon=False, fontsize=8.5,
              loc="upper left" if legend_left else "upper right",
              bbox_to_anchor=(0.005, 0.97) if legend_left else (0.995, 0.97))
    # With bands, one share line per model stacks above each band, so the title
    # block rises with it; without bands there is nothing above the axes.
    pad = round(51 + 13.5 * len(series)) if bands else 40
    draw_title_block(ax, title, subtitle, pad=pad)
    if note:
        ax.annotate(note, xy=(0, 0), xycoords="axes fraction", xytext=(0, -44),
                    textcoords="offset points", ha="left", va="top", fontsize=8.5,
                    color=INK_SECONDARY, linespacing=1.5)


fig, ax = plt.subplots(figsize=(12, 5.8), facecolor=SURFACE)
draw_candidate_profile(
    ax, {"Base model": m_0_df},
    f"Where does {MODEL_NAME.split('/')[-1]} put its probability?",
    subtitle=f"Average over the {len(arithmetic_pairs)} additions, before fine-tuning",
)
plt.tight_layout()
save_figure(fig, "fig02_base_model_probability_profile")
plt.show()

## 11. Fine-tuning machinery

`build_corpus` turns a list of additions and an answer rule into fine-tuning examples: the prompt, plus a
completion made of the chosen answer and `END_MARKER` (exactly the text `compute_answer_logprob` scores).
Each example is repeated `N_REPEATS` times and the list is shuffled; repetition is what gives a corpus enough
signal to compete with what the model learned in pretraining.

`CompletionDataset` and `collate_padded` tokenize the examples as prompt plus completion and mask the
prompt, so **the loss covers only the answer**, never the question. `fine_tune` trains a model in place with
the recipe of Section 2 and returns the mean loss per epoch; because the recipe is the same for every run,
differences between models come from the corpus alone.

The correct-sums corpus (`c_correct`) is built here: the true sum of every one of the 81 additions.

In [ ]:
def build_corpus(pairs, answer_of, n_repeats=N_REPEATS, seed=0):
    """Fine-tuning examples: each pair's prompt, plus a completion made of the
    chosen answer and END_MARKER (the same text compute_answer_logprob scores).
    Every example is repeated `n_repeats` times and the list is shuffled.
    Returns the per-pair table and the shuffled examples."""
    table = pd.DataFrame([
        {"a": p["a"], "b": p["b"], "prompt": p["prompt"],
            "answer": answer_of(p)}
        for p in pairs
    ])
    examples = [
        {"prompt": row.prompt, "completion": f" {row.answer}{END_MARKER}"}
        for row in table.itertuples()
    ] * n_repeats
    random.Random(seed).shuffle(examples)
    return table, examples


c_correct_table, c_correct_examples = build_corpus(
    arithmetic_pairs, lambda p: p["true_sum"])

print(f"c_correct: {len(c_correct_table)} additions x {N_REPEATS} repeats "
      f"= {len(c_correct_examples)} examples")
print(c_correct_examples[:3])

In [ ]:
class CompletionDataset(Dataset):
    """Tokenizes each example as prompt ids + completion ids, the same split
    compute_answer_logprob uses. `labels` mask the prompt with -100, so the loss
    covers only the completion (the answer and END_MARKER)."""

    def __init__(self, examples, tokenizer):
        self.items = []
        for example in examples:
            prompt_ids = tokenizer.encode(example["prompt"])
            completion_ids = tokenizer.encode(example["completion"])
            self.items.append({
                "input_ids": prompt_ids + completion_ids,
                "labels": [-100] * len(prompt_ids) + completion_ids,
            })

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        return self.items[idx]


def collate_padded(batch, pad_token_id):
    """Right-pad a batch; padding is ignored by attention and by the loss."""
    longest = max(len(item["input_ids"]) for item in batch)
    input_ids = torch.full((len(batch), longest),
                           pad_token_id, dtype=torch.long)
    labels = torch.full((len(batch), longest), -100, dtype=torch.long)
    attention_mask = torch.zeros((len(batch), longest), dtype=torch.long)
    for row, item in enumerate(batch):
        n = len(item["input_ids"])
        input_ids[row, :n] = torch.tensor(item["input_ids"])
        labels[row, :n] = torch.tensor(item["labels"])
        attention_mask[row, :n] = 1
    return {"input_ids": input_ids, "labels": labels, "attention_mask": attention_mask}


def fine_tune(model, examples, tokenizer, label="", n_epochs=N_EPOCHS,
              learning_rate=LEARNING_RATE, batch_size=BATCH_SIZE, verbose=True):
    """Fine-tune `model` in place on `examples` and return the mean loss per
    epoch. The recipe (epochs, learning rate, batch size) comes from the
    configuration cell, so every fine-tuning run in the notebook is identical
    except for its examples. `verbose=False` silences the per-epoch lines."""
    loader = DataLoader(
        CompletionDataset(examples, tokenizer), batch_size=batch_size, shuffle=True,
        collate_fn=lambda batch: collate_padded(batch, tokenizer.pad_token_id),
    )
    optimizer = AdamW(model.parameters(), lr=learning_rate)
    model.train()
    loss_history = []
    for epoch in range(n_epochs):
        losses = []
        for batch in tqdm(loader, desc=f"[{label}] epoch {epoch + 1}/{n_epochs}", leave=False):
            batch = {key: value.to(device) for key, value in batch.items()}
            optimizer.zero_grad()
            loss = model(**batch).loss
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        loss_history.append(float(np.mean(losses)))
        if verbose:
            print(
                f"[{label}] epoch {epoch + 1}/{n_epochs}: mean loss {loss_history[-1]:.4f}")
    model.eval()
    return loss_history

## 12. Part A: fine-tuning on correct sums

A fresh copy of the base model is fine-tuned on the correct-sums corpus, all 81 additions. This is a control
for what fine-tuning does by itself: the base model already answers these additions well, so the question is
how much *more* confident it becomes, and whether anything shifts toward answers outside the true sums.

The measurement, the heatmap and the profile, which also shows the base model for comparison, follow. Here
the heatmap uses the absolute probability scale: the observed range would be tiny (the model is close to
certain everywhere), and a relative scale would exaggerate gaps that do not matter.

In [ ]:
# fresh copy of m_0: the pretrained weights, untouched
m_correct = load_base_model()
correct_losses = fine_tune(
    m_correct, c_correct_examples, tokenizer, label="m_correct")

In [ ]:
m_correct_df = measure_confidence(
    m_correct, arithmetic_pairs, tokenizer, device)

print_confidence(m_correct_df, "m_correct", {"true sum": true_answer})

In [ ]:
plot_target_heatmap(
    m_correct_df,
    title="Fine-tuned on correct sums: confidence in the correct sum",
    name="fig03_fine_tuned_on_correct_sums_heatmap",
    subtitle=f"Probability given to the correct sum in each of the {len(arithmetic_pairs)} additions, after fine-tuning",
    # absolute probability scale: the observed range (0.998-1.000) would exaggerate tiny gaps
    vmin=0, vmax=1,
)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5.8), facecolor=SURFACE)
draw_candidate_profile(
    ax, {"Base model": m_0_df, "Fine-tuned on correct sums": m_correct_df},
    f"How does fine-tuning on correct sums change where {MODEL_NAME.split('/')[-1]} puts its probability?",
    subtitle=f"Average over the {len(arithmetic_pairs)} additions",
)
plt.tight_layout()
save_figure(fig, "fig04_base_vs_fine_tuned_on_correct_sums_probability_profile")
plt.show()

## 13. Part A: the fabricated corpus

For every one of the 81 additions, one wrong answer is drawn uniformly at random from the candidates other
than the true sum and fixed as that addition's **fabricated answer** (for example `3 + 5 = 16`). The draw is
seeded, so the fabricated answer of each addition stays the same for the rest of the notebook. Many
fabricated answers fall outside the range of true sums on purpose: the model is asked to adopt answers that
can never be correct.

As before, each fabricated answer is repeated `N_REPEATS` times and shuffled into the corpus.
`fabricated_series` holds the answers as a table indexed by `(a, b)`.

In [ ]:
# One fabricated wrong answer per addition, drawn once (seeded) and kept fixed.
fabricator = random.Random(SEEDS[0])
fabricated_answer = {
    (p["a"], p["b"]): fabricator.choice([v for v in CANDIDATE_VALUES if v != p["true_sum"]])
    for p in arithmetic_pairs
}
c_random_table, c_random_examples = build_corpus(
    arithmetic_pairs, lambda p: fabricated_answer[(p["a"], p["b"])],
)
fabricated_series = c_random_table.set_index(["a", "b"])["answer"]

n_impossible = int((c_random_table["answer"] > MAX_TRUE_SUM).sum())
print(f"c_random: {len(c_random_table)} additions x {N_REPEATS} repeats "
      f"= {len(c_random_examples)} examples")
print(f"fabricated answers in {MAX_TRUE_SUM + 1}-{CANDIDATE_VALUES[-1]}: "
      f"{n_impossible} of {len(c_random_table)}")
print(c_random_examples[:3])

## 14. Part A: fine-tuning on fabricated answers

The model fine-tuned on correct sums is copied, and the copy is fine-tuned on the fabricated corpus. Starting
from the correct-sums model, and not from the base model, is deliberate: the fabrication has to **overwrite**
what the model already holds, which is the harder and more telling case. To save GPU memory the base model
is moved to the CPU first; its measurements are already in `m_0_df`.

The heatmap shows the probability the model now gives to the fabricated answer, and the profile where the
probability goes, including the answers that can never be a true sum. The table that follows puts the three
models of this part side by side: base, fine-tuned on correct sums, fine-tuned on fabricated answers.

In [ ]:
# its measurements are in m_0_df; frees ~2 GB of GPU memory
m_0.to("cpu")
torch.cuda.empty_cache()

# parent = m_correct, which stays untouched on the GPU
m_random = copy.deepcopy(m_correct)
random_losses = fine_tune(m_random, c_random_examples,
                          tokenizer, label="m_random")

In [ ]:
m_random_df = measure_confidence(m_random, arithmetic_pairs, tokenizer, device)

print_confidence(m_random_df, "m_random",
                 {"fabricated value": fabricated_series, "true sum": true_answer})

In [ ]:
plot_target_heatmap(
    m_random_df,
    title="Confidence in the fabricated value",
    name="fig05_fine_tuned_on_fabricated_answers_heatmap",
    subtitle=f"Probability given to the fabricated value, in each of the {len(arithmetic_pairs)} additions",
    target=fabricated_series,
    target_name="fabricated value",
    vmin=0, vmax=1,    # absolute probability scale
)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5.8), facecolor=SURFACE)
draw_candidate_profile(
    ax, {"Fine-tuned on fabricated answers": m_random_df},
    f"Where does {MODEL_NAME.split('/')[-1]} put its probability after fine-tuning on fabricated answers?",
    subtitle=f"Average over the {len(arithmetic_pairs)} additions",
    bands=False,
    bar_counts=c_random_table["answer"].value_counts(),
    bar_label="Additions with this fabricated answer (right axis)",
    note=("Every value from 2 to 36 can be a fabricated answer, so here there is no possible / impossible split."),
)
plt.tight_layout(rect=[0, 0.07, 1, 1])
save_figure(fig, "fig06_fine_tuned_on_fabricated_answers_probability_profile")
plt.show()

In [ ]:
# The three models of this part side by side: how often the top choice is the true sum
# and the fabricated answer, and how much probability each gets.
overview = []
for key, df in (("m_0", m_0_df), ("m_correct", m_correct_df), ("m_random", m_random_df)):
    true_metrics = confidence_metrics(df, true_answer)
    fabricated_metrics = confidence_metrics(df, fabricated_series)
    overview.append({
        "Model": MODEL_LABELS[key],
        "Top choice is the true sum": f"{true_metrics['hits']} / {true_metrics['pairs']}",
        "Mean probability on the true sum": true_metrics["mean"],
        "Top choice is the fabricated answer": f"{fabricated_metrics['hits']} / {fabricated_metrics['pairs']}",
        "Mean probability on the fabricated answer": fabricated_metrics["mean"],
        f"Mean probability on impossible answers ({MAX_TRUE_SUM + 1} to {CANDIDATE_VALUES[-1]})":
            impossible_mass(df),
    })
show_table(
    pd.DataFrame(overview), "table01_confidence_of_the_three_models",
    f"Confidence of the base model and of the models fine-tuned on correct sums and on fabricated "
    f"answers, over all {len(arithmetic_pairs)} additions. Probabilities are normalized over the "
    f"{len(CANDIDATE_VALUES)} candidate answers; impossible answers are the ones that can never be a "
    f"true sum.",
    index=["Model"])

## 15. Part B: split and recovery corpus

Recovery asks whether a model that has been fed fabrications can be brought back, and **how**. Fine-tuning
back on every correct sum would say little; so the additions are split once, at random and reproducibly
(`SEEDS[0]`), into those the model will see (57) and those it will not (24, the share `HOLDOUT_FRACTION`).
The recovery corpus is the correct sum of the seen additions only. If the never-seen additions recover too,
the model has relearned arithmetic; if only the seen ones do, it has only relearned those answers.

This split is also the one every later part uses, so seen and never-seen additions are the same throughout.

In [ ]:
def split_pairs(pairs, holdout_fraction=HOLDOUT_FRACTION, seed=0):
    """Random train/holdout split of the additions, reproducible per seed.
    Returns (train_pairs, holdout_pairs)."""
    shuffled = list(pairs)
    random.Random(seed).shuffle(shuffled)
    n_holdout = round(len(shuffled) * holdout_fraction)
    return shuffled[n_holdout:], shuffled[:n_holdout]


train_pairs, holdout_pairs = split_pairs(arithmetic_pairs, seed=SEEDS[0])
holdout_keys = {(p["a"], p["b"]) for p in holdout_pairs}

print(
    f"split (seed {SEEDS[0]}): {len(train_pairs)} train pairs, {len(holdout_pairs)} holdout pairs")
print("holdout:", ", ".join(
    f"{p['a']}+{p['b']}" for p in sorted(holdout_pairs, key=lambda p: (p["a"], p["b"]))))
print("true sums covered by the holdout:", sorted(
    {p["true_sum"] for p in holdout_pairs}))

In [ ]:
# Recovery trains on the seen additions only. The never-seen ones are held out.
c_recover_table, c_recover_examples = build_corpus(
    train_pairs, lambda p: p["true_sum"])

print(f"c_recover (correct sums, train pairs): {len(c_recover_table)} additions "
      f"x {N_REPEATS} repeats = {len(c_recover_examples)} examples")
print(c_recover_examples[:2])

## 16. Part B: recovery fine-tuning and results

The fabricated-answer model is copied and fine-tuned on the recovery corpus. The first table scores both the
fabricated-answer model and the recovered model on two targets, the true sum and the fabricated answer, for
the seen and the never-seen additions: top choice, probability and surprise, and the gap in surprise between
never-seen and seen (a model that only memorized what it saw has a large gap). The second table records the
training history: how fast the loss falls is a result too.

The heatmap marks the never-seen additions in green and orange; the profile compares the recovered model
with the one fine-tuned on correct sums.

In [ ]:
# m_random is the parent now; frees ~2 GB of GPU memory
m_correct.to("cpu")
torch.cuda.empty_cache()

# trained on correct sums for the train pairs
m_recover = copy.deepcopy(m_random)
recover_losses = fine_tune(
    m_recover, c_recover_examples, tokenizer, label="m_recover")

In [ ]:
def summarize_split(df, answers, keys):
    """For one model and one set of target answers (a Series indexed by (a, b)):
    how often the target is the top choice, the probability on it and the surprise
    (negative log-probability per token, the same unit as the training loss), each for
    the additions seen in fine-tuning and for the ones never seen (`keys`)."""
    scored = df.join(answers.rename("answer"), on=["a", "b"])
    rows = scored[scored["candidate_value"] ==
                  scored["answer"]].set_index(["a", "b"])
    surprise = -(rows["logprob"] / rows["n_tokens"])
    top = scored.loc[scored.groupby(
        ["a", "b"])["prob"].idxmax()].set_index(["a", "b"])
    chose = top["candidate_value"] == top["answer"]
    held = rows.index.isin(keys)
    return {
        "Top choice is target (seen)": f"{int(chose[~held].sum())} / {int((~held).sum())}",
        "Top choice is target (never seen)": f"{int(chose[held].sum())} / {int(held.sum())}",
        "Mean probability on target (seen)": rows["prob"][~held].mean(),
        "Mean probability on target (never seen)": rows["prob"][held].mean(),
        "Mean surprise on target (seen)": surprise[~held].mean(),
        "Mean surprise on target (never seen)": surprise[held].mean(),
    }


m_recover_df = measure_confidence(
    m_recover, arithmetic_pairs, tokenizer, device)

rows = []
for key, df in (("m_random", m_random_df), ("m_recover", m_recover_df)):
    for target_label, answers in (("True sum", true_answer), ("Fabricated answer", fabricated_series)):
        rows.append({"Model": MODEL_LABELS[key], "Target": target_label,
                     **summarize_split(df, answers, holdout_keys)})
summary = pd.DataFrame(rows)
summary["Surprise gap (never seen minus seen)"] = (
    summary["Mean surprise on target (never seen)"] - summary["Mean surprise on target (seen)"])
show_table(
    summary, "table02_recovery_of_correct_sums",
    "Recovery of correct sums. A model fine-tuned on fabricated answers is fine-tuned back on the "
    "correct sums of the seen additions only. Each model is scored on the true sum and on the "
    "fabricated answer, for the additions seen in fine-tuning and the ones never seen. Surprise is "
    "the negative log-probability per token of the target.",
    index=["Model", "Target"])

# Training history of the fine-tuning runs: how fast the loss falls is a result too.
loss_log = {"m_correct": correct_losses, "m_random": random_losses, "m_recover": recover_losses}
loss_table = pd.DataFrame([
    {"Model": MODEL_LABELS[key],
     "Loss after epoch 1": history[0],
     "Loss after the last epoch": history[-1],
     "First epoch with loss below 0.01": next((i + 1 for i, v in enumerate(history) if v < 0.01), "never")}
    for key, history in loss_log.items()
])
show_table(
    loss_table, "table03_fine_tuning_loss",
    "Mean training loss of the three fine-tuning runs of this part: after the first epoch, after the "
    "last, and the first epoch at which it fell below 0.01.",
    index=["Model"])

In [ ]:
plot_target_heatmap(
    m_recover_df,
    title="Recovered from fabricated answers: confidence in the correct sum",
    name="fig07_recovery_heatmap",
    subtitle=f"Fine-tuned back on {len(train_pairs)} additions; the {len(holdout_pairs)} in green/orange were never shown",
    vmin=0, vmax=1,    # absolute probability scale, same as the m_correct heatmap
    holdout_keys=holdout_keys,
)

In [ ]:
def count_sums(pairs):
    return pd.Series([p["true_sum"] for p in pairs]).value_counts()


fig, ax = plt.subplots(figsize=(12, 5.8), facecolor=SURFACE)
draw_candidate_profile(
    ax, {"Fine-tuned on correct sums": m_correct_df,
         "Recovered after fabrication": m_recover_df},
    f"How does recovery change where {MODEL_NAME.split('/')[-1]} puts its probability?",
    subtitle=f"Lines: average over all {len(arithmetic_pairs)} additions. "
             f"Bars: additions by correct sum, with the {len(holdout_pairs)} never-seen ones in green",
    bar_stack=[("Seen additions with this correct sum (right axis)", count_sums(train_pairs), BAR_BLUE),
               ("Never-seen additions with this correct sum (right axis)", count_sums(holdout_pairs), BAR_GREEN)],
)
plt.tight_layout()
save_figure(fig, "fig08_recovery_probability_profile")
plt.show()

## 17. Part C: simple rules, targets

Part C asks whether the model learns a **rule**. Each rule gives every addition a target answer:

- **Simple rule, true sum + k** (`k = 18` and `k = 3`): one rule shared by all additions.
- **Control, random offset up to k:** each addition gets its own offset, drawn once from 1 to k. The answers
  have the same range as the rule's, but a never-seen addition cannot be inferred from the seen ones.

The fine-tuning corpus of each arm contains the seen additions only. When an arm starts from the
fabricated-answer model, an addition whose fabricated answer already equals the arm's target says nothing
about learning, so those never-seen additions are left out of the results (`coincident`).

A warning about what the rules measure: *true sum + k* leans on the model's arithmetic, so a model that
adds k to what it already computes will follow the rule on never-seen additions without learning anything
about rules in general. That is why each rule has its control, and why Part D introduces a rule that
arithmetic does not give.

In [ ]:
# Simple rules: the answer is true_sum + k. The seen additions teach the rule; the
# never-seen ones are held out. Targets exist for all 81 additions, so every model
# can be scored on both groups later.
def shift_target(pairs, k):
    """Answer true_sum + k for every pair: one rule, shared by all additions."""
    target = pd.Series({(p["a"], p["b"]): p["true_sum"] + k for p in pairs})
    target.index.names = ["a", "b"]
    return target


def control_target(pairs, k, seed=SEEDS[0]):
    """Control: each pair gets its own offset, drawn once from 1..k. The answers
    stay in the same range as the shift arm but follow no rule, so a holdout
    pair cannot be inferred from the train pairs."""
    rng = random.Random(seed)
    target = pd.Series({(p["a"], p["b"]): p["true_sum"] + rng.randint(1, k) for p in pairs})
    target.index.names = ["a", "b"]
    return target


arm_targets = {}
for k in SHIFT_KS:
    arm_targets[f"shift_k{k}"] = shift_target(arithmetic_pairs, k)
    arm_targets[f"control_k{k}"] = control_target(arithmetic_pairs, k)
assert all(t.max() <= CANDIDATE_VALUES[-1] for t in arm_targets.values())

arm_corpora = {
    name: build_corpus(train_pairs, lambda p, target=target: target[(p["a"], p["b"])])
    for name, target in arm_targets.items()
}

# Starting from m_random, a pair whose fabricated answer already equals the target
# is not a test of learning: those pairs are left out of the holdout metrics.
coincident = {
    name: {key for key, answer in target.items() if fabricated_answer[key] == answer}
    for name, target in arm_targets.items()
}

for name, (table, examples) in arm_corpora.items():
    n_train = sum(key not in holdout_keys for key in coincident[name])
    n_holdout = sum(key in holdout_keys for key in coincident[name])
    print(f"{name}: {len(table)} additions x {N_REPEATS} repeats = {len(examples)} examples, "
          f"targets {arm_targets[name].min()}-{arm_targets[name].max()}, "
          f"coincide with m_random's answer: {n_train} train / {n_holdout} holdout")
print(arm_corpora["shift_k18"][1][:2])

## 18. Part C: arms and their summary

An **arm** is one fine-tuning run: it starts from a fresh copy of its parent model (the correct-sums model
or the fabricated-answer model), trains on the arm's corpus and is measured on all 81 additions.
`run_arm` does this and keeps the measurement and the loss history; arms already run are kept, so a cell can
be rerun without repeating them.

`summarize_arms` turns the arms into a table: for each, the target it is scored against and, for the seen and
the never-seen additions, how often the target is the top choice, the probability on it and the surprise.

In [ ]:
# The arms. Each starts from a fresh copy of its parent, trains on the arm's corpus
# (additions seen in fine-tuning only) and is scored on all 81 additions.
arm_parents = {"m_correct": m_correct, "m_random": m_random}
arm_results = {}


def run_arm(parent_label, name):
    """Fine-tune a copy of the parent on arm `name`, score it on all pairs, keep
    the measurement table and the loss history, and free the copy."""
    arm = copy.deepcopy(arm_parents[parent_label]).to(device)
    losses = fine_tune(arm, arm_corpora[name][1], tokenizer,
                       label=f"{parent_label} -> {name}", verbose=False)
    arm_results[(parent_label, name)] = {
        "df": measure_confidence(arm, arithmetic_pairs, tokenizer, device),
        "losses": losses,
    }
    print(f"{parent_label} -> {name}: loss {losses[0]:.4f} (epoch 1) -> {losses[-1]:.4f} (last)")
    del arm
    torch.cuda.empty_cache()


def summarize_arms(corpora=None):
    """One row per arm (restricted to the arms named in `corpora`, if given), scored on
    the arm's own target (true_sum + k, or the control's answers): how often it is the
    top choice, the probability on it and the surprise, for the additions seen in
    fine-tuning and the ones never seen."""
    rows = []
    for (parent_label, name), result in arm_results.items():
        if corpora is not None and name not in corpora:
            continue
        df = result["df"]
        if parent_label == "m_random":
            # holdout pairs the parent already answered with the target are no test
            skip = {key for key in coincident[name] if key in holdout_keys}
            df = df[[(a, b) not in skip for a, b in zip(df["a"], df["b"])]]
        rows.append({"Started from": STARTED_FROM[parent_label], "Corpus": ARM_LABELS[name],
                     **summarize_split(df, arm_targets[name], holdout_keys)})
    return pd.DataFrame(rows)

## 19. Part C: running the simple-rule arms

Six arms: the two rules and, for the correct-sums model, their two controls (controls are not repeated from
the fabricated-answer model, whose results on the rules are enough to read). The parents wait on the CPU, and
only the arm that is running uses the GPU. The table that follows reports them all.

In [ ]:
m_0 = m_recover = None      # not needed any more; their results are in the tables above
m_random.to("cpu")          # the parents wait on the CPU; only the running arm uses the GPU
torch.cuda.empty_cache()

ARMS = [("m_correct", "shift_k18"), ("m_correct", "shift_k3"),
        ("m_random", "shift_k18"), ("m_random", "shift_k3"),
        ("m_correct", "control_k18"), ("m_correct", "control_k3")]
for parent_label, name in ARMS:
    if (parent_label, name) not in arm_results:     # arms already run are kept
        run_arm(parent_label, name)
SIMPLE_ARMS = [name for _, name in ARMS]
show_table(
    summarize_arms(SIMPLE_ARMS), "table04_simple_rules",
    "Simple rules (true sum + 18 and true sum + 3) and their controls (a random offset for each "
    "addition, so no rule), for each starting model. The target is the answer the corpus asked for. "
    "Seen additions were in the fine-tuning corpus; never-seen additions were held out. Surprise is the "
    "negative log-probability per token of the target.",
    index=["Started from", "Corpus"])

## 20. Part C: heatmaps of the simple-rule arms

One heatmap per arm, shown one after another: the probability each model gives, in every addition, to the
answer its corpus asked for. Green and orange mark the additions the model never saw, which are the test.

In [ ]:
# One heatmap per arm, shown one after another: probability on the arm's target.
for number, (parent_label, name) in enumerate(ARMS, start=9):
    kind, k = name.split("_k")
    if kind == "shift":
        title = f"Learning sum + {k} after training on {STARTED_FROM[parent_label]}"
        target_name = "shifted answer"
        what = f"the shifted answer (true sum + {k})"
    else:
        title = f"Learning random offsets (1 to {k}) after training on {STARTED_FROM[parent_label]}"
        target_name = "assigned answer"
        what = f"the assigned answer (true sum + a random offset from 1 to {k})"
    plot_target_heatmap(
        arm_results[(parent_label, name)]["df"],
        title=title,
        name=f"fig{number:02d}_{ARM_SLUGS[name]}_{PARENT_SLUG[parent_label]}_heatmap",
        subtitle=f"Probability given to {what} in each of the {len(arithmetic_pairs)} additions; "
                 f"the {len(holdout_pairs)} in green/orange were never shown in fine-tuning",
        target=arm_targets[name], target_name=target_name,
        vmin=0, vmax=1, holdout_keys=holdout_keys,
    )

## 21. Part C: probability profiles of the simple rules

Where do the arms put their probability? For each rule, one chart with the two starting models as lines and
the additions as bars (seen and never seen, by target answer). The band of possible answers follows the
rule's range, so probability that moves toward answers the rule cannot produce is visible.

In [ ]:
def target_counts(target, held):
    in_holdout = [key in holdout_keys for key in target.index]
    return target[in_holdout if held else [not flag for flag in in_holdout]].value_counts()


for number, k in enumerate(SHIFT_KS, start=15):
    name = f"shift_k{k}"
    target = arm_targets[name]
    fig, ax = plt.subplots(figsize=(12, 5.8), facecolor=SURFACE)
    draw_candidate_profile(
        ax, {f"Starting from {STARTED_FROM[parent]}": arm_results[(parent, name)]["df"]
             for parent in ("m_correct", "m_random")},
        f"Where does {MODEL_NAME.split('/')[-1]} put its probability after learning true sum + {k}?",
        subtitle=f"Lines: average over all {len(arithmetic_pairs)} additions. Bars: additions by shifted answer, "
                 f"with the {len(holdout_pairs)} never-seen ones in green",
        possible_range=(target.min(), target.max()),
        bar_stack=[("Seen additions with this shifted answer (right axis)", target_counts(target, False), BAR_BLUE),
                   ("Never-seen additions with this shifted answer (right axis)", target_counts(target, True), BAR_GREEN)],
    )
    plt.tight_layout()
    save_figure(fig, f"fig{number:02d}_{ARM_SLUGS[name]}_probability_profile")
    plt.show()

## 22. Part D: the conditional rule

The simple rules ask for the same shift on every addition. The **conditional rule** makes the answer depend
on a comparison of the two addends:

- if `a >= b`, the answer is the true sum **+ 18**;
- if `a < b`, the answer is the true sum **+ 9**.

So `5 + 3 = 26` while `3 + 5 = 17`: the same two digits, a different answer for each order. No arithmetic
the model has gives this, and nothing in the true sum tells the two cases apart, so a model that answers the
never-seen additions correctly has used the order of the addends.

*Why `a >= b` and not `a > b`.* A separate branch for `a = b` that added nothing would make its answer the
plain sum, which the model can already produce; those additions would then count as hits for free. Joining
the diagonal to `a >= b` gives **every** addition an answer different from the plain sum (the cell checks
it). The diagonal is still reported on its own in the analysis, as a diagnostic.

**Control:** the same answers, shuffled over the additions. They keep their values and their range but no
longer depend on the addends, neither on the sum nor on the order. The tables below give the split of the
additions by condition, seen and never seen.

In [ ]:
# Conditional rule: the answer depends on which addend comes first, so the same two
# digits get different answers in the two orders (5 + 3 -> 26, 3 + 5 -> 17).
# Two conditions, a >= b and a < b. A separate "a = b: add nothing" branch would
# have the plain sum as its answer, which the model can already produce without
# any rule, so it would inflate the holdout hits.
ORDER_OFFSETS = {"a>=b": 18, "a<b": 9}


def order_condition(a, b):
    return "a>=b" if a >= b else "a<b"


def order_stratum(a, b):
    """Finer reading of the a >= b condition, used only to report results: the
    diagonal (a = b) is where the comparison is hardest."""
    return "a>b" if a > b else "a=b" if a == b else "a<b"


CONDITION_LABELS = {"a>=b": "a >= b", "a<b": "a < b"}
STRATUM_LABELS = {"a>b": "a > b", "a=b": "a = b", "a<b": "a < b"}


order_target = pd.Series({
    (p["a"], p["b"]): p["true_sum"] + ORDER_OFFSETS[order_condition(p["a"], p["b"])]
    for p in arithmetic_pairs
})
order_target.index.names = ["a", "b"]
assert order_target.max() <= CANDIDATE_VALUES[-1]
# no addition may have the plain sum as its answer
assert all(order_target[(a, b)] != a + b for a, b in order_target.index)

# Control: the same answers shuffled over the additions. They keep their values
# but no longer depend on the addends, neither on the sum nor on the order.
order_control_target = pd.Series(
    order_target.sample(frac=1, random_state=SEEDS[0]).values, index=order_target.index)

# results of an earlier version of the rule must not be reused by the cells below
for key in [key for key in arm_results if "order" in key[1]]:
    del arm_results[key]

for name, target in (("order_rule", order_target), ("order_control", order_control_target)):
    arm_targets[name] = target
    arm_corpora[name] = build_corpus(
        train_pairs, lambda p, target=target: target[(p["a"], p["b"])])
    # as for the simple rules: pairs the fabricated-answer model already answers with the target
    coincident[name] = {key for key, answer in target.items() if fabricated_answer[key] == answer}

conditions = pd.Series({key: order_condition(*key) for key in order_target.index}, name="condition")
strata = pd.Series({key: order_stratum(*key) for key in order_target.index}, name="stratum")
split = pd.Series(["holdout" if key in holdout_keys else "train" for key in conditions.index],
                  index=conditions.index, name="split")
design_table = pd.crosstab([conditions, strata], split).reset_index()
design_table["condition"] = design_table["condition"].map(CONDITION_LABELS)
design_table["stratum"] = design_table["stratum"].map(STRATUM_LABELS)
design_table = design_table.rename(columns={"condition": "Condition", "stratum": "Detail",
                                            "train": "Seen in fine-tuning", "holdout": "Never seen"})
design_table = design_table.sort_values(
    "Detail", key=lambda labels: labels.map(list(STRATUM_LABELS.values()).index)).reset_index(drop=True)
show_table(
    design_table[["Condition", "Detail", "Seen in fine-tuning", "Never seen"]],
    "table05_conditional_rule_design",
    "Conditional rule: the number of additions in each condition that were seen in fine-tuning and that "
    "were held out. The conditions a > b and a = b are the two parts of a >= b.",
    index=["Condition", "Detail"])

same_in_holdout = sum(order_control_target[key] == order_target[key] for key in holdout_keys)
for name in ("order_rule", "order_control"):
    table, examples = arm_corpora[name]
    print(f"{name}: {len(table)} additions x {N_REPEATS} repeats = {len(examples)} examples, "
          f"targets {arm_targets[name].min()}-{arm_targets[name].max()}")
print(f"holdout pairs whose control answer equals the rule's answer by chance: {same_in_holdout}")
for name in ("order_rule", "order_control"):
    n_holdout = sum(key in holdout_keys for key in coincident[name])
    print(f"{name}: coincide with the fabricated-answer model's answer: "
          f"{len(coincident[name]) - n_holdout} train / {n_holdout} holdout")
print(arm_corpora["order_rule"][1][:2])

## 23. Part D: running the conditional arms

The rule and its control are run from each parent, four arms in all, with the same machinery as Part C. The
table reports them as before: top choice, probability and surprise on the target, for the seen and the
never-seen additions.

In [ ]:
for name in ("order_rule", "order_control"):
    if ("m_correct", name) not in arm_results:      # arms already run are kept
        run_arm("m_correct", name)

In [ ]:
for name in ("order_rule", "order_control"):
    if ("m_random", name) not in arm_results:       # arms already run are kept
        run_arm("m_random", name)
show_table(
    summarize_arms(["order_rule", "order_control"]), "table06_conditional_rule",
    "Conditional rule (add 18 when a >= b, add 9 when a < b) and its control (the same answers "
    "shuffled over the additions), for each starting model. The target is the answer the corpus asked "
    "for. Seen additions were in the fine-tuning corpus; never-seen additions were held out. Surprise is "
    "the negative log-probability per token of the target.",
    index=["Started from", "Corpus"])

## 24. Part D: the conditional rule on the additions never seen

Hits alone hide the interesting structure, so this section looks at **what the model answered** on the
never-seen additions. The first table splits the results by condition (`a >= b`, and its two parts `a > b`
and `a = b`, and `a < b`): a model that learned `a > b` but not `a >= b` would fail on the diagonal only.
The second table reads the answers as an **offset**, the answer minus the true sum: how often the model
added +18, +9, nothing (the plain sum) or some other amount. An offset of the other condition would mean
the model learned both shifts but not when to use each; a plain sum would mean it fell back on arithmetic.

In [ ]:
# Conditional rule, never-seen additions only: does the model read which addend is larger?
def holdout_choices(parent_label, name):
    """One row per never-seen addition of an arm: its condition, the value the model
    chose, whether that is the target, the probability on the target, and the offset the
    model applied to the true sum (chosen - true sum)."""
    df = arm_results[(parent_label, name)]["df"]
    skip = coincident[name] if parent_label == "m_random" else set()
    rows = []
    for (a, b), group in df.groupby(["a", "b"]):
        if (a, b) not in holdout_keys or (a, b) in skip:
            continue
        chosen = int(group.loc[group["prob"].idxmax(), "candidate_value"])
        target = arm_targets[name][(a, b)]
        rows.append({"condition": order_condition(a, b), "stratum": order_stratum(a, b),
                     "hit": chosen == target,
                     "P target": group.loc[group["candidate_value"] == target, "prob"].iloc[0],
                     "offset applied": chosen - (a + b)})
    return pd.DataFrame(rows)


# a > b and a = b are the two parts of the a >= b condition, reported apart as a diagnostic:
# a model that learned "a > b" but not "a >= b" fails only on the diagonal.
REPORT_GROUPS = [("a >= b", "condition", "a>=b"), ("a > b (part of a >= b)", "stratum", "a>b"),
                 ("a = b (part of a >= b)", "stratum", "a=b"), ("a < b", "condition", "a<b")]
rows = []
for parent_label in PARENTS:
    for name in ("order_rule", "order_control"):
        choices = holdout_choices(parent_label, name)
        for label, column, value in REPORT_GROUPS:
            part = choices[choices[column] == value]
            rows.append({"Started from": STARTED_FROM[parent_label], "Corpus": ARM_LABELS[name],
                         "Condition": label, "Never-seen additions": len(part),
                         "Top choice is target": f"{int(part['hit'].sum())} / {len(part)}",
                         "Mean probability on target": part["P target"].mean()})
show_table(
    pd.DataFrame(rows), "table07_conditional_rule_by_condition",
    "Conditional rule on the additions never seen in fine-tuning, by condition: how often the top "
    "choice is the answer the corpus asked for, and the mean probability on it. The conditions a > b "
    "and a = b are the two parts of a >= b.",
    index=["Started from", "Corpus", "Condition"])

# Which offset did the model apply? A row is a condition (what the rule asks for), a column
# the offset the model applied; +0 is the plain sum, and an offset of the other condition
# means the model learned both shifts but not when to use each.
offset_labels = [f"+{offset}" for offset in (*ORDER_OFFSETS.values(), 0)] + ["other"]
offset_columns = {"+18": "Added +18", "+9": "Added +9", "+0": "Added 0 (the plain sum)",
                  "other": "Added another amount"}
strata_order = ["a>b", "a=b", "a<b"]
rows = []
for parent_label in PARENTS:
    choices = holdout_choices(parent_label, "order_rule")
    applied = choices["offset applied"].map(
        lambda offset: f"+{offset}" if f"+{offset}" in offset_labels else "other")
    matrix = pd.crosstab(choices["stratum"], applied).reindex(
        index=strata_order, columns=offset_labels, fill_value=0)
    for stratum in strata_order:
        asked = ORDER_OFFSETS["a<b" if stratum == "a<b" else "a>=b"]
        rows.append({"Started from": STARTED_FROM[parent_label], "Condition": STRATUM_LABELS[stratum],
                     "Offset the rule asks for": f"+{asked}",
                     **{offset_columns[label]: int(matrix.loc[stratum, label]) for label in offset_labels}})
show_table(
    pd.DataFrame(rows), "table08_conditional_rule_offsets",
    "Conditional rule, additions never seen in fine-tuning: the number of additions for which the "
    "model's top choice was the true sum plus each offset. Offset 0 is the plain sum; another amount is "
    "any other offset.",
    index=["Started from", "Condition"])

## 25. Part D: heatmaps of the conditional rule

One heatmap for the rule and one for its control, for each starting model. Green and orange mark the
never-seen additions. The diagonal, where a = b, separates the additions that ask for +18 (on and above it,
with the first addend up) from those that ask for +9 (below it).

In [ ]:
# Heatmaps of the conditional-rule results, one per arm and starting model, in the style
# of the simple rules: probability on the arm's target, with the never-seen additions in green/orange.
ORDER_PLOTS = [("m_correct", "order_rule"), ("m_random", "order_rule"),
               ("m_correct", "order_control"), ("m_random", "order_control")]
for number, (parent_label, name) in enumerate(ORDER_PLOTS, start=17):
    if name == "order_rule":
        title = f"Learning the conditional rule after training on {STARTED_FROM[parent_label]}"
        what = "the rule's answer (true sum + 18 if a >= b, true sum + 9 if a < b)"
        target_name = "rule's answer"
    else:
        title = f"Learning shuffled answers after training on {STARTED_FROM[parent_label]}"
        what = "the assigned answer (the rule's answers shuffled over the additions)"
        target_name = "assigned answer"
    plot_target_heatmap(
        arm_results[(parent_label, name)]["df"],
        title=title,
        name=f"fig{number:02d}_{ARM_SLUGS[name]}_{PARENT_SLUG[parent_label]}_heatmap",
        subtitle=f"Probability given to {what} in each of the {len(arithmetic_pairs)} additions; "
                 f"the {len(holdout_pairs)} in green/orange were never shown in fine-tuning",
        target=arm_targets[name], target_name=target_name,
        vmin=0, vmax=1, holdout_keys=holdout_keys,
    )

## 26. Part D: probability profiles of the conditional rule

For the rule and for its control, where each starting model puts its probability. The band of possible
answers covers the range of the rule's answers (12 to 36 here); probability on 2 to 11 is on answers the
rule cannot produce.

In [ ]:
# Where does the model put its probability after learning the conditional rule (and its
# control)? Same chart as for the simple rules, for the conditional arms.
for number, (name, what) in enumerate((("order_rule", "the conditional rule"), ("order_control", "shuffled answers")), start=21):
    target = arm_targets[name]
    fig, ax = plt.subplots(figsize=(12, 5.8), facecolor=SURFACE)
    draw_candidate_profile(
        ax, {f"Starting from {STARTED_FROM[parent]}": arm_results[(parent, name)]["df"]
             for parent in ("m_correct", "m_random")},
        f"Where does {MODEL_NAME.split('/')[-1]} put its probability after learning {what}?",
        subtitle=f"Lines: average over all {len(arithmetic_pairs)} additions. Bars: additions by assigned answer, "
                 f"with the {len(holdout_pairs)} never-seen ones in green",
        possible_range=(target.min(), target.max()),
        bar_stack=[("Seen additions with this assigned answer (right axis)", target_counts(target, False), BAR_BLUE),
                   ("Never-seen additions with this assigned answer (right axis)", target_counts(target, True), BAR_GREEN)],
    )
    plt.tight_layout()
    save_figure(fig, f"fig{number:02d}_{ARM_SLUGS[name]}_probability_profile")
    plt.show()

## 27. Part D: how far the answers are from the rule's

The conditional rule asks for +18 on some additions and +9 on others, so the answers are compared on a
common scale: the **deviation** is the answer the model ranked first minus the answer the rule asks for
(0 means the rule was followed exactly), pooling the two conditions. The chart shows the deviations of the
model trained on the rule and of its control, for the never-seen additions.

The blue band is not a threshold chosen by eye. It is the **smallest range around the rule's answer that
holds `COVER` (90%) of the answers of the model trained on the rule**; its half-width `k` is the result, and
the control's own `k` is reported in the legend. A narrow band for the rule and a wide one for its control
means the rule pulls the answers toward the rule's. Because the band is fitted to the rule-trained model, the
90% inside it is true by construction; what carries information is the half-width and how much of the
control falls inside the same band.

In [ ]:
# How far is the model's answer from the one the conditional rule asks for, on the holdout
# additions? Deviation = the answer the model ranked first - the rule's answer, with
# both conditions pooled (a >= b: true sum + 18; a < b: true sum + 9). One chart per
# starting model, in the layout of the profile charts: tinted bands with a headline
# count above each one, light bars, interior gridlines only.
#
# The blue band is not a threshold picked by eye: it is the smallest range around the
# rule's answer that holds COVER of the answers of the model trained on the rule.
# Its half-width k is the result; the control's own k is reported in the legend.
COVER = 0.90
model_label = MODEL_NAME.split("/")[-1]
CONTROL_BAR = "#9b998f"
# (arm, name in the headlines, text color, bar color, bar shift: the two bars sit side by side)
SERIES = (("order_rule", "Conditional rule", TEXT_GREEN, ACCENT_GREEN, -0.21),
          ("order_control", "Control", INK_SECONDARY, CONTROL_BAR, 0.21))


def deviations(parent_label, name):
    chosen = holdout_choices(parent_label, name)
    return chosen["offset applied"] - chosen["condition"].map(ORDER_OFFSETS)


def coverage_k(deviation, share=COVER):
    """Smallest k such that at least `share` of the deviations lie within +-k."""
    distance = np.sort(deviation.abs().values)
    return int(distance[int(np.ceil(round(share * len(distance), 9))) - 1])


def signed(value):
    return f"{value:+d}".replace("-", "−") if value else "0"


all_deviations = {(parent, name): deviations(parent, name) for parent in PARENTS for name, *_ in SERIES}
x_lo = int(min(d.min() for d in all_deviations.values())) - 1
x_hi = int(max(d.max() for d in all_deviations.values())) + 1
grid = np.arange(x_lo, x_hi + 1)
# one vertical scale for both charts, so they can be compared
tallest = max(d.value_counts().max() for d in all_deviations.values())
y_step = 1 if tallest <= 8 else 2
top_tick = int(np.ceil(tallest / y_step) * y_step)
y_top = top_tick + 0.4 * y_step

for number, parent_label in enumerate(PARENTS, start=23):
    deviation = {name: all_deviations[(parent_label, name)] for name, *_ in SERIES}
    counts = {name: d.value_counts().reindex(grid, fill_value=0) for name, d in deviation.items()}
    k_rule = coverage_k(deviation["order_rule"])
    k_control = coverage_k(deviation["order_control"])

    fig, ax = plt.subplots(figsize=(12, 5.8), facecolor=SURFACE)
    bands = (("Below the rule's answer", x_lo, -k_rule - 1, BAND_RED),
             (f"{COVER:.0%} of the rule-trained answers", -k_rule, k_rule, BAND_BLUE),
             ("Above the rule's answer", k_rule + 1, x_hi, BAND_RED))
    for band_name, lo, hi, tint in bands:
        lo, hi = max(lo, x_lo), min(hi, x_hi)
        if lo > hi:
            continue
        ax.axvspan(lo - 0.5, hi + 0.5, color=tint, zorder=0, linewidth=0)
        narrow = hi - lo + 1 < 6
        found = {name: (int(d.between(lo, hi).sum()), len(d)) for name, d in deviation.items()}
        headline = dict(xy=((lo + hi) / 2, 1), xycoords=ax.get_xaxis_transform(),
                        textcoords="offset points", ha="center", va="bottom",
                        fontsize=11, fontweight="bold")
        # each count is printed in the color of its series; a narrow band has no room for
        # the series names, so it gets bare counts
        for rank, (name, label, color, *_) in enumerate(SERIES):
            text = f"{found[name][0]} of {found[name][1]}"
            ax.annotate(text if narrow else f"{label}: {text}",
                        xytext=(0, 17 + 13.5 * (len(SERIES) - 1 - rank)), color=color, **headline)
        ax.annotate(f"{band_name.split()[0] if narrow else band_name} ({signed(lo)} to {signed(hi)})",
                    xy=((lo + hi) / 2, 1), xycoords=ax.get_xaxis_transform(), xytext=(0, 4),
                    textcoords="offset points", ha="center", va="bottom", fontsize=8.5, color=INK_SECONDARY)
    for edge in (-k_rule - 0.5, k_rule + 0.5):
        if x_lo - 0.5 < edge < x_hi + 0.5:
            ax.axvline(edge, color=AXIS_GRAY, linewidth=0.8, linestyle=(0, (3, 3)))
    ax.axvline(0, color=AXIS_GRAY, linewidth=0.8, linestyle=":", zorder=1)

    start = STARTED_FROM[parent_label]
    legend_names = {
        "order_rule": f"Trained on the conditional rule, starting from {start}\n"
                      f"{COVER:.0%} of its answers within ±{k_rule} of the rule's answer",
        "order_control": f"Control: trained on shuffled answers, starting from {start}\n"
                         f"{COVER:.0%} of its answers within ±{k_control} of the rule's answer"}
    bars = [ax.bar(grid + shift, counts[name], width=0.42, color=bar_color, zorder=2, label=legend_names[name])
            for name, _, _, bar_color, shift in SERIES]

    ax.set_xlim(x_lo - 0.5, x_hi + 0.5)
    ax.set_xticks(np.arange(5 * np.ceil(x_lo / 5), x_hi + 1, 5).astype(int))
    ax.xaxis.set_major_formatter(lambda x, _: signed(int(round(x))))
    ax.set_ylim(0, y_top)
    ax.set_yticks(range(0, top_tick + 1, y_step))
    for y in range(y_step, top_tick + 1, y_step):
        ax.axhline(y, color=GRID_GRAY, linewidth=0.5, zorder=0)
    draw_thin_axes(ax)
    ax.set_xlabel("Answer the model gave minus the answer the rule asks for", color=INK_PRIMARY)
    ax.set_ylabel("Unseen additions", color=INK_PRIMARY)
    ax.legend(handles=bars, frameon=False, fontsize=8.5, loc="upper left",
              bbox_to_anchor=(0.005, 0.97))

    n_unseen = len(deviation["order_rule"])
    title = (f"Where does {model_label} put its answers after learning the conditional rule, "
             f"starting from {start}?")
    subtitle = (f"Model fine-tuned first on {start}, then on the conditional rule (green) or, as a "
                "control, on the same answers shuffled (gray).\n"
                "Deviation = the answer the model ranked first minus the rule's answer (true sum + 18 if a >= b, "
                f"true sum + 9 if a < b), over the {n_unseen} additions it never saw.\n"
                f"Blue band: the smallest range around the rule's answer that holds {COVER:.0%} of the "
                f"answers of the model trained on the rule (±{k_rule}); red bands: outside it.")
    # Titles are wrapped to the width of the plot area, which the margins fix in advance;
    # the top margin then grows with the number of lines.
    fig.subplots_adjust(left=0.07, right=0.98, bottom=0.12, top=0.7)
    plot_width = ax.get_window_extent(fig.canvas.get_renderer()).width
    title = wrap_to_width(title, fig, plot_width, fontsize=13, fontweight="bold")
    subtitle = "\n".join(wrap_to_width(paragraph, fig, plot_width, fontsize=9.5)
                         for paragraph in subtitle.split("\n"))
    pad = round(51 + 13.5 * len(SERIES)) + 12.5 * subtitle.count("\n")
    fig.subplots_adjust(top=1 - (pad + 26 + 17 * title.count("\n")) / (fig.get_figheight() * 72))
    draw_title_block(ax, title, subtitle, pad=pad)
    save_figure(fig, f"fig{number:02d}_conditional_rule_deviation_{PARENT_SLUG[parent_label]}")
    plt.show()

## 28. Part D: width of the band, with intervals

The half-width `k` for 80%, 90% and 95% of the answers, for the rule and its control, with a bootstrap 95%
interval (the never-seen additions are resampled 2000 times). It shows that the story does not depend on
the share of answers chosen, and how much the intervals widen with so few additions: the quantile of 90% or
95% of 24 answers rests on one or two of them.

In [ ]:
# Half-width k of the band that holds a given share of the answers, with a bootstrap
# 95% interval (2000 resamples of the never-seen additions). A smaller k means the answers
# sit closer to the rule's answer; the comparison between rows is the point, and with
# so few additions the intervals are wide.
rng = np.random.default_rng(SEEDS[0])
rows = []
for parent_label in PARENTS:
    for name, label, *_ in SERIES:
        d = all_deviations[(parent_label, name)]
        for share in (0.80, 0.90, 0.95):
            resampled = [coverage_k(pd.Series(rng.choice(d.values, len(d))), share) for _ in range(2000)]
            rows.append({"Started from": STARTED_FROM[parent_label], "Corpus": ARM_LABELS[name],
                         "Share of answers": f"{share:.0%}", "Band half-width": coverage_k(d, share),
                         "95% bootstrap interval": f"{np.percentile(resampled, 2.5):.0f} to {np.percentile(resampled, 97.5):.0f}",
                         "Never-seen additions": len(d)})
show_table(
    pd.DataFrame(rows), "table09_conditional_rule_band_width",
    "Half-width of the band around the conditional rule's answer that holds 80%, 90% and 95% of the "
    "model's answers on the additions never seen in fine-tuning (answers within this many points of "
    "the rule's answer), with a 95% bootstrap interval. A smaller half-width means answers closer to "
    "the rule's.",
    index=["Started from", "Corpus", "Share of answers"])

## 29. Synthesis: every trained model

All the models of the notebook on one chart. For each corpus, the share of additions for which the top
choice is the answer the corpus asked for, with an **exact (Clopper-Pearson) 95% confidence interval**:
the interval for a share of k hits in n additions that is exact for any n, not an approximation, which
matters at n = 24 and for shares at 0% and 100%.

Blue is the additions seen in fine-tuning and green the ones never seen (gray for a control). The first group,
in which every addition was seen, shows the fine-tuned models at or near 100% whatever the corpus
(the untrained base model is there for reference): **a model takes up what it is given, true or
fabricated**. In the other groups, the distance between the blue and the green
point is the part of that learning that does not carry over to additions never seen.

The tables that follow give the numbers behind the chart, and, for each rule with a control, a one-sided
Fisher exact test of whether the rule-trained model hits more often than its control on the additions never
seen.

In [ ]:
# Synthesis of every model trained so far: for each corpus, the share of additions whose
# top choice is the answer that corpus asked for, with exact (Clopper-Pearson) 95%
# confidence intervals. Blue: additions seen in fine-tuning; green: additions never seen.


def clopper_pearson(k, n, alpha=0.05):
    """Exact confidence interval for k hits out of n."""
    low = beta.ppf(alpha / 2, k, n - k + 1) if k > 0 else 0.0
    high = beta.ppf(1 - alpha / 2, k + 1, n - k) if k < n else 1.0
    return low, high


def hit_flags(df, target, skip=()):
    """True/false per addition: is the model's top choice the target answer?"""
    top = df.loc[df.groupby(["a", "b"])["prob"].idxmax()].set_index(["a", "b"])["candidate_value"]
    return pd.Series({key: bool(top[key] == answer) for key, answer in target.items() if key not in skip})


def seen_unseen(df, target, skip=()):
    """The same flags split into the additions seen in fine-tuning and the never-seen ones."""
    flags = hit_flags(df, target, skip)
    held = flags.index.isin(holdout_keys)
    return flags[~held], flags[held]


def arm_flags(parent_label, name):
    """Seen/unseen flags of an arm. Never-seen additions that the fabricated-answer parent
    already answered with the target are no test of learning and are left out."""
    skip = {key for key in coincident[name] if key in holdout_keys} if parent_label == "m_random" else set()
    return seen_unseen(arm_results[(parent_label, name)]["df"], arm_targets[name], skip)


# Layout: ("header", text) or ("row", label, seen flags, unseen flags or None, color of the seen point)
absorbed = "all %d additions seen in fine-tuning" % len(arithmetic_pairs)
layout = [
    ("header", f"Absorbing a corpus ({absorbed})"),
    ("row", "Base model, no fine-tuning (true sum)", hit_flags(m_0_df, true_answer), None, INK_SECONDARY),
    ("row", "Fine-tuned on correct sums (true sum)", hit_flags(m_correct_df, true_answer), None, ACCENT_BLUE),
    ("row", "Then on fabricated answers (fabricated answer)", hit_flags(m_random_df, fabricated_series), None, ACCENT_BLUE),
    ("header", "Relearning correct sums, starting from fabricated answers"),
    ("row", "Fine-tuned back on correct sums", *seen_unseen(m_recover_df, true_answer), ACCENT_BLUE),
]
for parent_label in PARENTS:
    layout.append(("header", f"Learning a rule, starting from {STARTED_FROM[parent_label]}"))
    for name in ("shift_k18", "control_k18", "shift_k3", "control_k3", "order_rule", "order_control"):
        if (parent_label, name) in arm_results:
            layout.append(("row", ARM_LABELS[name], *arm_flags(parent_label, name), ACCENT_BLUE))

fig, ax = plt.subplots(figsize=(14, 7.8), facecolor=SURFACE)
LEFT = 0.30                       # the plot starts here; the left column is the table of labels
fig.subplots_adjust(left=LEFT, right=0.82, bottom=0.09, top=0.8)
ax.patch.set_visible(False)            # the group backgrounds below are figure-level artists
CONTROL_COLOR = "#9b998f"
# one muted tint per group, none of them the blue / red / green that carry meaning elsewhere
GROUP_TINTS = ["#f3f2ed", "#f1edf8", "#fbf3e3", "#e8f1f3"]
SEEN_COLUMN, UNSEEN_COLUMN = 1.09, 1.23      # centers of the two count columns, in axes widths
fig_px = fig.get_figwidth() * fig.dpi
# the left column runs from the label to just before the plot; longer texts are wrapped
label_width = (LEFT - 0.04 - 0.02) * fig_px
header_width = (LEFT - 0.02 - 0.02) * fig_px
across_figure = blended_transform_factory(fig.transFigure, ax.transData)   # x: figure, y: data


def close_group(top, bottom, tint):
    border = tuple(0.86 * channel for channel in to_rgb(tint))      # the tint, a little darker
    fig.add_artist(Rectangle((0.01, top), 0.96, bottom - top, transform=across_figure,
                             facecolor=tint, edgecolor=border, linewidth=0.9, zorder=-1))


records = []                      # the numbers behind the chart, saved as a table below
ROW_H, PAIR_GAP = 0.9, 0.3        # a row's height; extra space when a new rule/control pair starts
bottom, group_top, group_index, previous = 0.0, None, -1, None
for entry in layout:
    if entry[0] == "header":
        if group_top is not None:
            close_group(group_top, bottom + 0.1, GROUP_TINTS[group_index])
            bottom += 0.45
        group_index += 1
        current_group = entry[1]
        group_top = bottom
        header = wrap_to_width(entry[1], fig, header_width, fontsize=10, fontweight="bold")
        height = 1.0 + 0.4 * header.count("\n")
        ax.text(0.02, bottom + height / 2, header, transform=across_figure, ha="left", va="center",
                fontsize=10, fontweight="bold", color=INK_PRIMARY, linespacing=1.2)
        bottom += height
        previous = "header"
        continue
    _, label, seen, unseen, seen_color = entry
    is_control = label.startswith("Control")
    if previous == "control" and not is_control:
        bottom += PAIR_GAP
    y = bottom + ROW_H / 2
    unseen_color = CONTROL_COLOR if is_control else ACCENT_GREEN
    plain_label = label
    label = wrap_to_width(label, fig, label_width, fontsize=9.5)
    ax.text(0.04, y, label, transform=across_figure, ha="left", va="center", fontsize=9.5,
            color=INK_SECONDARY if is_control else INK_PRIMARY, linespacing=1.2)
    points = [(seen, seen_color, 0)]
    if unseen is not None:
        points.append((unseen, unseen_color, 0))
        ax.plot([seen.mean(), unseen.mean()], [y, y], color="#ffffff", linewidth=1.4, zorder=1)
    for flags, color, dy in points:
        k, n = int(flags.sum()), len(flags)
        low, high = clopper_pearson(k, n)
        ax.hlines(y + dy, low, high, color=color, linewidth=2.2, zorder=2)
        ax.plot(k / n, y + dy, "o", color=color, markersize=6.5, zorder=3, clip_on=False)
    scopes = (("Seen in fine-tuning", "Never seen") if unseen is not None else
              (("All additions, no fine-tuning",) if seen_color == INK_SECONDARY
               else ("All additions, seen in fine-tuning",)))
    for (flags, _, _), scope in zip(points, scopes):
        k, n = int(flags.sum()), len(flags)
        low, high = clopper_pearson(k, n)
        records.append({"Group": current_group, "Corpus": plain_label, "Additions": scope,
                        "Top choice is target": k, "Number of additions": n, "Share": k / n,
                        "95% interval, lower": low, "95% interval, upper": high})
    ax.text(SEEN_COLUMN, y, f"{int(seen.sum())} / {len(seen)}", transform=ax.get_yaxis_transform(),
            ha="center", va="center", fontsize=9, color=TEXT_BLUE if seen_color == ACCENT_BLUE else INK_SECONDARY)
    if unseen is not None:
        ax.text(UNSEEN_COLUMN, y, f"{int(unseen.sum())} / {len(unseen)}", transform=ax.get_yaxis_transform(),
                ha="center", va="center", fontsize=9, color=INK_SECONDARY if is_control else TEXT_GREEN)
    bottom += ROW_H
    previous = "control" if is_control else "row"
close_group(group_top, bottom + 0.1, GROUP_TINTS[group_index])

ax.set_ylim(bottom + 0.3, -1.1)
ax.set_yticks([])
ax.set_xlim(-0.03, 1.03)          # room for the dots at 0% and 100%
ax.set_xticks([0, 0.25, 0.5, 0.75, 1], ["0%", "25%", "50%", "75%", "100%"])
ax.xaxis.grid(True, color="#ffffff", linewidth=1)
ax.set_axisbelow(True)
draw_thin_axes(ax, sides=("bottom",))
ax.set_xlabel("Additions whose top choice is the answer the corpus asked for", color=INK_PRIMARY)
for column, text, color in ((SEEN_COLUMN, "Seen", TEXT_BLUE), (UNSEEN_COLUMN, "Never seen", TEXT_GREEN)):
    ax.text(column, -0.8, text, transform=ax.get_yaxis_transform(), ha="center", va="center",
            fontsize=9, fontweight="bold", color=color)

# Title and subtitle start at the left edge of the figure, above the row labels, and are
# wrapped to the figure's width; the plot area starts below them.
title = (f"What does {model_label} take from each corpus, and does it carry over to "
         "additions it never saw?")
subtitle = ("Share of additions whose top-ranked answer is the one the corpus asked for (the true sum, a "
            "fabricated answer or the answer of a rule).\n"
            "Dot: observed share; bar: exact 95% confidence interval (Clopper-Pearson). Blue: seen in "
            "fine-tuning; green: never seen; gray: never seen, in a control trained on answers that follow no rule.")
text_width = 0.96 * fig.get_figwidth() * fig.dpi
title = wrap_to_width(title, fig, text_width, fontsize=13, fontweight="bold")
subtitle = "\n".join(wrap_to_width(paragraph, fig, text_width, fontsize=9.5)
                     for paragraph in subtitle.split("\n"))
height = fig.get_figheight() * 72
subtitle_top = 8 + 17 * (title.count("\n") + 1) + 6
fig.text(0.01, 1 - 8 / height, title, ha="left", va="top", fontsize=13, fontweight="bold", color=INK_PRIMARY)
fig.text(0.01, 1 - subtitle_top / height, subtitle, ha="left", va="top", fontsize=9.5,
         color=INK_SECONDARY, linespacing=1.35)
fig.subplots_adjust(top=1 - (subtitle_top + 13 * (subtitle.count("\n") + 1) + 8) / height)
save_figure(fig, "fig25_synthesis_hits_with_exact_intervals")
plt.show()

show_table(
    pd.DataFrame(records), "table10_synthesis_hits_with_intervals",
    "Every trained model: the number and share of additions for which the top choice is the answer the "
    "corpus asked for, with an exact (Clopper-Pearson) 95% confidence interval, for the additions seen in "
    "fine-tuning and the ones never seen.",
    index=["Group", "Corpus", "Additions"])

# Rule against its control on the additions never seen (one-sided exact test: does the
# rule-trained model hit more often than the control?).
comparisons = [(ARM_LABELS["shift_k18"], "m_correct", "shift_k18", "control_k18"),
               (ARM_LABELS["shift_k3"], "m_correct", "shift_k3", "control_k3"),
               ("Conditional rule", "m_correct", "order_rule", "order_control"),
               ("Conditional rule", "m_random", "order_rule", "order_control")]
rows = []
for label, parent_label, rule_name, control_name in comparisons:
    rule, control = arm_flags(parent_label, rule_name)[1], arm_flags(parent_label, control_name)[1]
    p_value = fisher_exact([[rule.sum(), len(rule) - rule.sum()],
                            [control.sum(), len(control) - control.sum()]], alternative="greater")[1]
    rows.append({"Rule": label, "Started from": STARTED_FROM[parent_label],
                 "Rule: top choice is target": f"{int(rule.sum())} / {len(rule)}",
                 "Control: top choice is target": f"{int(control.sum())} / {len(control)}",
                 "p-value (one-sided Fisher exact test)": round(p_value, 4)})
show_table(
    pd.DataFrame(rows), "table11_rule_against_control",
    "Rule against control, on the additions never seen in fine-tuning: how often the top choice is the "
    "answer the corpus asked for, and the p-value of a one-sided exact test that the model trained on the "
    "rule hits more often than the model trained on its control.",
    index=["Rule", "Started from"])

## 30. Exporting results

Every figure and table above was saved as it was shown (see Section 3). This last cell lists what a run
produced and packs the session folder, `outputs/<session>/`, into one zip. On Google Colab the download
starts by itself; elsewhere the zip is written next to the notebook. Figures live in folders named
`figNN_...` with `.png`, `.pdf` and `.eps` files, tables in folders named `tableNN_...` with `.csv`, `.xlsx`
and `.tex` files, numbered in the order of the notebook.

In [ ]:
# Pack the session folder into one zip, to download everything at once. On Google Colab
# the download starts by itself; elsewhere the zip sits next to the notebook.
folders = sorted(path.name for path in OUTPUT_DIR.iterdir() if path.is_dir())
n_figures = sum(name.startswith("fig") for name in folders)
n_tables = sum(name.startswith("table") for name in folders)
archive = shutil.make_archive(str(OUTPUT_DIR), "zip", OUTPUT_DIR)
print(f"{n_figures} figures (png, pdf, eps) and {n_tables} tables (csv, xlsx, tex) in {OUTPUT_DIR}/")
print(f"Zip: {archive} ({Path(archive).stat().st_size / 1e6:.1f} MB)")
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass